# Snowflake

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/snowflake/snowflake.ipynb)

Snowflake is a fully managed cloud data platform. You load data into it, query it with SQL, and pay for storage and compute separately. This notebook explains each Snowflake feature a data engineer uses, from the architecture to loading, transformation, governance, cost and tuning.

Every topic follows the same order: **explanation** in simple words, **use cases**, **limitations and things to remember**, and then a **code example**. Snowflake SQL is shown in `sql` blocks; it needs a Snowflake account, so it is not run here. Where a runnable cell helps, it uses [DuckDB](https://duckdb.org) as a local stand-in: Parquet row groups play the role of micro-partitions, and Time Travel and streams are simulated so the mechanics are visible.

The notebook has six parts:

1. **Architecture and warehouses**
2. **Storage and query performance**
3. **Protecting and loading data**
4. **Transformation, governance and cost**
5. **Newer features and the tuning playbook**
6. **Practice and real-world cases**

*Last reviewed: November 2026, against the Snowflake documentation as of November 2026.*

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"duckdb": "duckdb>=1.4"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## What Snowflake is: storage, compute and services

### Explanation
A traditional database ties disks and CPUs to one server, so heavy loads slow down dashboards and you pay for peak
capacity all day. Snowflake solves this by splitting the system into three layers that scale and bill independently.

```text
            +-----------------------------------------------+
            | Cloud services: parse, optimize, metadata,     |
            | transactions, result cache                     |
            +-----------------------------------------------+
               |                |                 |
         [ bi_wh ]        [ elt_wh ]        [ adhoc_wh ]     <- compute (virtual warehouses)
               |                |                 |
            +-----------------------------------------------+
            | Storage: micro-partitions in cloud object store |
            +-----------------------------------------------+
```

**Storage** holds your tables as micro-partitions and is billed per terabyte per month. **Compute** is a set of
virtual warehouses, clusters of servers billed in credits per second of uptime. A credit is Snowflake's unit of
compute billing. **Cloud services** is the always-on "brain": it compiles queries, keeps the metadata that makes
pruning possible, manages transactions and serves the result cache.

The layers coordinate through cloud services, not through each other. Storage keeps many versions of each
micro-partition, and the transaction manager records which version belongs to which commit. A warehouse receives a
plan that already names the partition versions to read, so two warehouses never share locks or cache state. A commit
becomes visible to every warehouse as soon as the new version is published. Your job shifts from tuning servers to
shaping queries and sizing warehouses.

### Use cases
- **Workload isolation:** give BI, ELT and ad-hoc users separate warehouses so a backfill never slows a dashboard.
- **Elastic backfills:** resize the ELT warehouse up for one heavy run and back down, without moving any data.
- **Cost attribution:** read per-warehouse credits and per-feature serverless credits to charge teams back.
- **Bill investigation:** compare consumed and billed cloud-services credits when the invoice looks odd.

### Limitations and things to remember
- Cloud services is billed only for the part of a day's usage that exceeds **10 percent of that day's warehouse
  compute credits**. Ordinary parsing, optimization and metadata work is effectively free.
- The 10 percent allowance is a daily, account-wide calculation. A day with little warehouse compute has a tiny
  allowance, so a metadata-heavy day can show billed cloud-services credits without any warehouse running.
- Serverless features (Snowpipe, automatic clustering, materialized-view maintenance) sit outside the allowance and
  are billed as serverless credits attributed to each feature.
- `ACCOUNT_USAGE` views show all consumed cloud-services credits; `METERING_DAILY_HISTORY` shows what was billed. Do
  not add `CREDITS_USED_CLOUD_SERVICES` from `QUERY_HISTORY` straight onto the bill. `ACCOUNT_USAGE` views also lag
  behind real time.
- Scale: a MEDIUM warehouse (4 credits/hour) running 8 hours a day for 22 days uses 4 x 8 x 22 = 704 credits. The
  bill moves with total warehouse uptime and bytes read per statement.
- Complex queries can spend noticeable `COMPILATION_TIME` in cloud services before any warehouse work begins.
- Isolation is a compute property, not a cache property: an identical query on a second warehouse can be answered
  from the shared result cache, so a two-warehouse benchmark may measure the cache instead of the cluster.

### Code example

The code checks where you are connected, gives each workload its own warehouse, and reads billed credits per service.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Where am I connected, and with what compute?
SELECT CURRENT_ACCOUNT(), CURRENT_REGION(), CURRENT_ROLE(), CURRENT_WAREHOUSE();

-- One warehouse per workload; each has its own size and lifecycle over the same data.
CREATE WAREHOUSE bi_wh    WAREHOUSE_SIZE = 'SMALL'  AUTO_SUSPEND = 300 AUTO_RESUME = TRUE;
CREATE WAREHOUSE elt_wh   WAREHOUSE_SIZE = 'LARGE'  AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;
CREATE WAREHOUSE adhoc_wh WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;

-- Serverless credits are attributed to the feature that used them.
SELECT service_type, SUM(credits_used) AS credits
FROM snowflake.account_usage.metering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY service_type
ORDER BY credits DESC;
```

## Virtual warehouses

### Explanation
Queries need CPU and memory, and different jobs need very different amounts. A **virtual warehouse** is a metered
compute cluster that you create, size, start and stop. It runs your SQL against shared storage, and you pay credits
per second while it is running.

Each size step doubles the credit rate:

```text
XSMALL 1   SMALL 2   MEDIUM 4   LARGE 8   XLARGE 16    (credits per hour)
2XLARGE 32   3XLARGE 64   4XLARGE 128   5XLARGE 256   6XLARGE 512
```

A bigger warehouse changes speed, not the amount of data a query must read. Think of it like adding cooks to a
kitchen: more cooks finish a big order faster, but they cannot help if one dish must be stirred by one person.
Scale-up only speeds the parts of a plan that can be split across nodes. A skewed join key, a single-threaded
aggregation or one very wide row leaves extra nodes idle.

A warehouse is also a pool of worker threads. When more statements arrive than it can run at once, the extra ones
wait in a **queue**, and `QUEUED_OVERLOAD_TIME` records that wait separately from `EXECUTION_TIME`. Queueing is a
concurrency symptom, not a SQL symptom: a query that takes two seconds alone can wait thirty seconds behind others.

To size one, start small, measure p95 latency, and move up one size only when a measured query is latency-bound and
its slow operator is parallel.

### Use cases
- **Heavy backfill:** bump `transform_wh` from MEDIUM to LARGE for the run, then back down.
- **BI latency tuning:** raise `bi_wh` one size when p95 dashboard latency is unacceptable, and measure again.
- **End of batch window:** suspend the ELT warehouse explicitly when the last job finishes.
- **Workload split:** separate warehouses for ELT, BI and data science so each is sized and billed on its own.

### Limitations and things to remember
- Scaling up is credit-neutral only when runtime halves: 600 s on MEDIUM = 4 x 600 / 3600 = 0.667 credits; 300 s on
  LARGE = 8 x 300 / 3600 = 0.667 credits. If runtime falls by less than the rate rises, you lose money.
- XSMALL takes about four times as long as MEDIUM on parallel work but uses nearly the same credits.
- A resize applies to statements that start after it completes; running queries keep their old size. Resize between
  batches, not in the middle of one.
- In a multi-cluster warehouse the rate applies per cluster, so six clusters burn six times the listed rate.
- Fix pruning and query shape before paying for a bigger warehouse; size does not reduce bytes scanned.

### Code example

The code creates a suspended warehouse, resizes it for a backfill, and compares credits with p95 latency.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE WAREHOUSE transform_wh
  WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';   -- heavy backfill
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'MEDIUM';  -- back to steady state
ALTER WAREHOUSE transform_wh SUSPEND;                        -- end of the batch window

-- Credits and p95 latency per warehouse: is a bigger size worth it?
SELECT q.warehouse_name,
       ROUND(APPROX_PERCENTILE(q.execution_time, 0.95) / 1000, 1) AS p95_exec_s,
       ROUND(SUM(q.queued_overload_time) / 1000, 1)               AS queued_s
FROM snowflake.account_usage.query_history q
WHERE q.start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY q.warehouse_name
ORDER BY p95_exec_s DESC;
```

## Multi-cluster warehouses

### Explanation
A single warehouse queues statements when too many arrive at once, for example at 9 a.m. when every dashboard
refreshes. A **multi-cluster warehouse** fixes this by adding identical clusters when queries wait and removing them
when load drops. Each cluster is a full copy of the warehouse at the same size.

`MIN_CLUSTER_COUNT` and `MAX_CLUSTER_COUNT` set the range. `MAX_CONCURRENCY_LEVEL` is the number of concurrent
queries one cluster is expected to absorb; a lower value makes Snowflake add the next cluster sooner.

The **scaling policy** decides how eagerly clusters are added:

```text
STANDARD : queries queue -> add a cluster now      (low latency, interactive BI)
ECONOMY  : wait for proof the cluster stays busy   (fewer start/stops, batch)
```

STANDARD is reactive and removes clusters when load drops. ECONOMY is deliberate: a burst may queue slightly longer,
but the warehouse avoids starting and stopping clusters repeatedly.

The rule of thumb: scale **up** (bigger size) for one slow query, scale **out** (more clusters) for many queued
queries. Multi-cluster helps when queue time, not per-query scan cost, is the bottleneck.

### Use cases
- **Morning dashboard storm:** `bi_wh` with 1-6 clusters and STANDARD absorbs the refresh peak, then shrinks.
- **Shared reporting warehouse:** `reporting_wh` with 2-10 clusters and ECONOMY for many scheduled reports.
- **Concurrency diagnosis:** count queries with `QUEUED_OVERLOAD_TIME > 0` to decide whether scale-out is worth it.

### Limitations and things to remember
- Each cluster bills at the full per-hour rate, so `MAX_CLUSTER_COUNT` is a cost multiplier as well as a capacity
  setting.
- Adding clusters never makes one query faster; every query runs inside one cluster. A slow query with no queueing
  needs a different fix.
- Clusters are removed only down to `MIN_CLUSTER_COUNT`; a minimum above 1 always pays for those clusters while
  running.
- The 60-second billing minimum on each start applies per cluster.
- Watch `AVG_RUNNING` and `AVG_QUEUED_LOAD` from `WAREHOUSE_LOAD_HISTORY` to see cluster demand over time.

### Code example

The code configures STANDARD and ECONOMY multi-cluster warehouses and measures queueing per warehouse.

Snowflake SQL (shown, not run in this notebook):

```sql
ALTER WAREHOUSE bi_wh SET
  MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 6 SCALING_POLICY = 'STANDARD';    -- interactive

ALTER WAREHOUSE reporting_wh SET
  MIN_CLUSTER_COUNT = 2 MAX_CLUSTER_COUNT = 10 SCALING_POLICY = 'ECONOMY';    -- batch

-- Queueing is the signal that scale-out is worth trying.
SELECT warehouse_name,
       COUNT(*) AS queries,
       COUNT_IF(queued_overload_time > 0)         AS queued_queries,
       ROUND(SUM(queued_overload_time) / 1000, 1) AS total_queued_s
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('hour', -6, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY total_queued_s DESC;
```

## Auto-suspend and auto-resume

### Explanation
A running warehouse burns credits even when no query is running. **Auto-suspend** stops the warehouse after a set
number of seconds of inactivity. **Auto-resume** starts it again automatically when the next statement arrives.
Together they mean you pay only while work is happening, like lights on a motion sensor.

The trade-off is the cache. A warehouse keeps recently read data on its local SSD (the warehouse cache). Suspending
clears that cache, so the first query after a resume reads from remote storage and is slower.

```text
last query ---- idle 60 s ----> suspend (billing stops, SSD cache lost)
next query ---> auto-resume (billed at least 60 s) ---> cold cache run
```

Match the value to how likely the next query is: about 60 seconds for scheduled batch, several minutes (300-600) for
interactive bursts where a warm cache matters, and never 0.

Example: a SMALL warehouse (2 credits/hour) left running through a 14-hour overnight gap burns 2 x 14 = 28 idle
credits. With `AUTO_SUSPEND = 60` it uses almost nothing after the last statement, at the cost of a cold cache on the
next run.

### Use cases
- **Scheduled ELT:** `elt_wh` with `AUTO_SUSPEND = 60` stops right after each batch.
- **Interactive BI:** `bi_wh` with `AUTO_SUSPEND = 600` keeps the cache warm between analysts' clicks.
- **Idle-cost audit:** find warehouses with many credits but few active hours in `WAREHOUSE_METERING_HISTORY`.
- **Change history:** check `WAREHOUSE_EVENTS_HISTORY` to see who resumed, suspended or resized a warehouse.

### Limitations and things to remember
- Each start is billed for at least 60 seconds, so a warehouse that wakes for a five-second statement pays a minute.
- A multi-cluster warehouse pays that 60-second minimum per cluster.
- Values below 60 seconds are effectively the 60-second minimum on a standard warehouse; aggressive numbers give no
  benefit.
- `AUTO_SUSPEND = 0` means never suspend, and the warehouse burns idle credits.
- Keep auto-resume on, or scheduled work against a suspended warehouse fails instead of running.
- Resuming means the local SSD cache starts empty.

### Code example

The code sets suspend values per workload and finds warehouses that may be burning idle credits.

Snowflake SQL (shown, not run in this notebook):

```sql
ALTER WAREHOUSE bi_wh        SET AUTO_SUSPEND = 600 AUTO_RESUME = TRUE;  -- interactive bursts
ALTER WAREHOUSE elt_wh       SET AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;  -- scheduled batch
ALTER WAREHOUSE reporting_wh SET AUTO_SUSPEND = 0;                        -- anti-pattern: idle credits

-- Many credits over few active hours points to idle time.
SELECT warehouse_name,
       SUM(credits_used) AS credits,
       COUNT(DISTINCT DATE_TRUNC('hour', start_time)) AS active_hours
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -3, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;
```

## Caching layers

### Explanation
Reading the same data again and again wastes time and credits. Snowflake has three caches, each at a different
layer, that avoid repeated work.

```text
layer            cache             lives where            survives suspend?  lifetime
cloud services   result cache      account-wide           yes                24 h, cleared by DML
cloud services   metadata cache    account-wide           yes                persistent
compute          warehouse cache   one warehouse's SSD    no                 until suspend
```

The **result cache** stores the final result of a query. It is keyed by the query text, the role and its privileges,
relevant session parameters, and the versions of the tables involved. It lives in cloud services and needs no
warehouse, so a hit returns in milliseconds even while the warehouse is suspended, and it is free.

The **warehouse cache** holds raw micro-partition bytes on the nodes' local SSD. It belongs to one warehouse and is
lost when that warehouse suspends.

The **metadata cache** holds table statistics such as row counts and per-partition min/max values. It is shared
account-wide and survives suspension.

Any DML on a table creates a new table version, so the next run of the same query text executes again.

### Use cases
- **Dashboard refreshes:** identical tile queries on unchanged tables return from the result cache at no cost.
- **Benchmarking:** change the query text or table between runs so you measure compute, not the cache.
- **Tuning auto-suspend:** a longer suspend keeps the warehouse cache warm for repeated interactive scans.
- **Cache audit:** read `PERCENTAGE_SCANNED_FROM_CACHE` in `QUERY_HISTORY` before assuming a query was cached.

### Limitations and things to remember
- Result-cache entries last 24 hours and are invalidated by DML on any table in the query.
- Non-deterministic functions such as `CURRENT_TIMESTAMP()` make a query ineligible for the result cache.
- The result cache is shared across warehouses, so a second warehouse can be answered without scanning anything.
- A result-cache hit scans nothing, so `PARTITIONS_SCANNED` and `PERCENTAGE_SCANNED_FROM_CACHE` can both be zero on
  a hit. Check `TOTAL_ELAPSED_TIME` and whether a warehouse started before concluding the cache missed.
- `PERCENTAGE_SCANNED_FROM_CACHE` measures the warehouse SSD cache, not the result cache.

### Code example

The code shows when the result cache answers a query and when DML forces it to run again.

Snowflake SQL (shown, not run in this notebook):

```sql
SELECT COUNT(*) FROM analytics.orders;                    -- runs, result stored
SELECT COUNT(*) FROM analytics.orders;                    -- identical text, unchanged table: cache hit
SELECT CURRENT_TIMESTAMP(), COUNT(*) FROM analytics.orders;  -- non-deterministic: never result-cached
INSERT INTO analytics.orders (order_id) VALUES (999999);  -- new table version
SELECT COUNT(*) FROM analytics.orders;                    -- re-executed despite identical text

-- Was each run served from the warehouse SSD cache, and how long did it take?
SELECT query_id, total_elapsed_time / 1000 AS elapsed_s,
       percentage_scanned_from_cache, partitions_scanned
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE 'SELECT COUNT(*) FROM analytics.orders%'
ORDER BY start_time DESC
LIMIT 5;
```

## Micro-partitions and pruning

### Explanation
Scanning a whole multi-terabyte table to find one day of data is slow and expensive. Snowflake avoids this with
micro-partitions and pruning.

A **micro-partition** is a 50-500 MB block of rows that Snowflake stores and manages for you. For every column in
every micro-partition, Snowflake records statistics such as the minimum and maximum value. **Pruning** means skipping
any micro-partition whose min/max range cannot match the query's filter, using only that metadata.

Snowflake fills partitions in the order rows arrive and closes one when it reaches the target size. So the order of
an insert becomes the physical order of the table:

```text
unordered load                     ordered load
P1  2026-01-01 .. 2026-12-31       P1  2026-01-01 .. 2026-01-07
P2  2026-01-01 .. 2026-12-31       P2  2026-01-08 .. 2026-01-14
P3  2026-01-01 .. 2026-12-31       P3  2026-01-15 .. 2026-01-21
P4  2026-01-01 .. 2026-12-31       P4  2026-01-22 .. 2026-01-28
one-day filter -> 4 partitions      one-day filter -> 1 partition
```

Partitions are immutable: Snowflake never rewrites one in place. An update or delete writes the new row images to a
new partition and marks the old one superseded. Because statistics exist for every column, a filter on any single
column can prune. The whole performance model is simple: load rows ordered by your common filter, and keep the filter
column free of functions.

### Use cases
- **Date-filtered fact tables:** sort loads by `order_date` so monthly reports read a few partitions.
- **Slow-query triage:** compare `PARTITIONS_SCANNED` with `PARTITIONS_TOTAL` in `QUERY_HISTORY`.
- **Predicate review:** rewrite `CAST(order_date AS STRING) LIKE '2026-09%'` as a date range so it prunes.

### Limitations and things to remember
- You do not define micro-partitions; Snowflake creates and manages them.
- Functions, casts and type mismatches on the filter column hide the literal from pruning and force a scan.
- Updates break the ordering a sorted load created: rewritten rows land in new partitions that may span the whole
  date range, so a heavily updated table drifts back toward full scans.
- Restore order by reclustering (a clustering key) or by reloading in sorted order.
- Pruning helps only when the filter is selective and the column's values are well ordered across partitions.

### Code example

The code loads rows in filter order, contrasts a pruning predicate with a non-pruning one, and measures the scan ratio.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Ordered load: partition ranges stay narrow.
INSERT INTO analytics.orders (order_id, order_date, customer_id, amount)
SELECT order_id, order_date, customer_id, amount
FROM staging.orders_raw
ORDER BY order_date, customer_id;

-- Prunes: a literal range on the raw column.
SELECT COUNT(*) FROM analytics.orders WHERE order_date BETWEEN '2026-09-01' AND '2026-09-30';

-- Does not prune: the cast hides the literal.
SELECT COUNT(*) FROM analytics.orders WHERE CAST(order_date AS STRING) LIKE '2026-09%';

-- Percent of partitions scanned by recent queries.
SELECT ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned
FROM snowflake.account_usage.query_history
WHERE query_text ILIKE '%analytics.orders%'
ORDER BY start_time DESC
LIMIT 5;
```

## Clustering keys

### Explanation
Natural load order works until updates and late-arriving data scatter values across partitions, and pruning stops
working. A **clustering key** is one or more columns you declare so that Snowflake keeps the table physically
ordered by them. **Automatic clustering** is the serverless background service that does the reordering.

Snowflake measures disorder with **clustering depth**: the average number of partitions that must be probed for a
point in the key space. Lower is better, and about 1 means near-perfect ordering.
`SYSTEM$CLUSTERING_INFORMATION` reports `average_depth` and `average_overlaps`.

```text
key (order_date, customer_id)
before reclustering:  average_depth = 3.4   -> a point lookup probes about 3 partitions
after reclustering:   average_depth = 1.2   -> about 1 partition
```

Automatic clustering compares current depth with the declared key and rewrites overlapping partitions until order
is restored. It does not run on a schedule; Snowflake triggers it as disorder builds up, so cost arrives in bursts
after large loads.

Use a clustering key only on large tables that are filtered repeatedly by the same columns. Lead the key with the
column used in range filters, usually a date, then add a high-cardinality id. Judge the key by depth and query speed
against the serverless credits it consumes.

### Use cases
- **Large event table:** cluster `analytics.events` by `(event_date, customer_id)` for date-range reports.
- **Update-heavy facts:** let automatic clustering restore the order that continuous updates destroy.
- **Frozen history:** suspend reclustering on a table that has stopped changing.
- **Cost review:** sum `AUTOMATIC_CLUSTERING_HISTORY` credits per table to drop keys that do not pay off.

### Limitations and things to remember
- A key helps predicates on its leading columns. A key of `(order_date, customer_id)` may not prune a filter on
  `customer_id` alone, because date ranges of neighboring partitions still overlap.
- Automatic clustering bills serverless credits, in bursts after large loads.
- Not worth it for small tables or tables rarely filtered by the key columns.
- `SUSPEND RECLUSTERING` pauses maintenance; `DROP CLUSTERING KEY` removes the key.
- Calling `SYSTEM$CLUSTERING_INFORMATION` without a key reports on the table's current clustering key.

### Code example

The code declares a clustering key, checks its depth, pauses maintenance, and reads its serverless cost.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Date first for range filters, then a high-cardinality id.
ALTER TABLE analytics.events CLUSTER BY (event_date, customer_id);

-- Depth and overlaps: lower is better, about 1 is near-perfect.
SELECT SYSTEM$CLUSTERING_INFORMATION('analytics.events', '(event_date, customer_id)');

ALTER TABLE analytics.events SUSPEND RECLUSTERING;   -- table stopped changing
ALTER TABLE analytics.events RESUME RECLUSTERING;
ALTER TABLE analytics.events DROP CLUSTERING KEY;    -- remove entirely

-- Serverless credits spent on automatic clustering, per table.
SELECT table_name, SUM(credits_used) AS clustering_credits
FROM snowflake.account_usage.automatic_clustering_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY table_name
ORDER BY clustering_credits DESC;
```

The runnable cell below simulates micro-partition pruning and clustering depth with DuckDB. It writes the same rows twice to Parquet, once in arrival order and once sorted by `event_date`, and treats each Parquet row group as a micro-partition with min/max statistics.

In [ ]:
import datetime as dt
import pathlib
import tempfile

import duckdb

ROW_COUNT = 300_000
ROWS_PER_PARTITION = 4096
DAY_SPAN = 300
WEEK_START, WEEK_END = dt.date(2026, 6, 1), dt.date(2026, 6, 7)

clustering_con = duckdb.connect()
clustering_dir = pathlib.Path(tempfile.mkdtemp())

# Random-looking dates in arrival order; each Parquet row group plays the role of a micro-partition.
events_sql = f"""
    SELECT i AS event_id,
           DATE '2026-01-01' + CAST((i * 7919) % {DAY_SPAN} AS INTEGER) AS event_date,
           i % 50 AS amount
    FROM range({ROW_COUNT}) t(i)
"""
for layout, order_by in (("arrival", ""), ("sorted", "ORDER BY event_date")):
    clustering_con.execute(
        f"COPY ({events_sql} {order_by}) TO '{clustering_dir}/{layout}.parquet' "
        f"(FORMAT parquet, ROW_GROUP_SIZE {ROWS_PER_PARTITION})"
    )


def partition_profile(path: str, lo: dt.date, hi: dt.date) -> tuple[int, int, float]:
    """Return partitions scanned for [lo, hi], total partitions, and average clustering depth."""
    ranges = clustering_con.execute(f"""
        SELECT CAST(stats_min_value AS DATE), CAST(stats_max_value AS DATE)
        FROM parquet_metadata('{path}') WHERE path_in_schema = 'event_date'
    """).fetchall()
    scanned = sum(1 for p_lo, p_hi in ranges if p_hi >= lo and p_lo <= hi)
    # Depth: average number of partitions whose range overlaps each partition's range.
    depth = sum(
        sum(1 for o_lo, o_hi in ranges if o_lo <= p_hi and o_hi >= p_lo) for p_lo, p_hi in ranges
    ) / len(ranges)
    return scanned, len(ranges), depth


profiles = {
    layout: partition_profile(f"{clustering_dir}/{layout}.parquet", WEEK_START, WEEK_END)
    for layout in ("arrival", "sorted")
}
for layout, (scanned, total, depth) in profiles.items():
    print(f"{layout:>7}: partitions scanned {scanned}/{total}, average depth {depth:.1f}")

arrival_scanned, arrival_total, _ = profiles["arrival"]
sorted_scanned, _, sorted_depth = profiles["sorted"]
assert arrival_scanned == arrival_total and sorted_scanned <= 3 and sorted_depth <= 3

## Search optimization service

### Explanation
Min/max pruning cannot help a lookup like `WHERE email = 'someone@example.com'`: every partition's email range
covers that value, so Snowflake scans them all. The **search optimization service** solves this by maintaining a
persistent access path, an index-like structure, alongside the table for chosen columns and predicate methods.

The optimizer rewrites a matching predicate to use that structure instead of scanning micro-partitions, so your query
text stays the same and the results do not change. It is like the index at the back of a book: you jump to the right
page instead of reading every page.

It targets predicates that pruning cannot narrow:

- equality and `IN` on a column
- `LIKE` with a leading wildcard (`'%/checkout/complete%'`)
- `ARRAY_CONTAINS`
- geospatial predicates

You enable it per column and per method (`EQUALITY`, `SUBSTRING`, and so on), not for the whole table. Maintenance
runs asynchronously on serverless compute: after DML, the structure catches up in the background.

### Use cases
- **Customer lookup by email:** a support tool fetches one customer from a very large table.
- **URL search in clickstream:** find events whose `url` contains a path, with a leading wildcard.
- **ID point lookups:** fetch a few rows by `customer_id` from a table clustered on date.

### Limitations and things to remember
- It does not speed up aggregations, joins or broad range scans.
- Maintenance runs on every DML, so cost is proportional to change volume; on a high-churn staging table it can cost
  more than it saves.
- Improvements are not instant after DML; the structure catches up in the background.
- Enable it on large, slowly changing lookup tables, and check the query profile to confirm the access path is used.
- Remove it with `DROP SEARCH OPTIMIZATION` when the lookups that justified it disappear.

### Code example

The code enables search optimization on selected columns, runs queries that benefit, and removes it again.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Enable on specific columns and methods, not the whole table.
ALTER TABLE analytics.customers ADD SEARCH OPTIMIZATION ON EQUALITY(email, customer_id);
ALTER TABLE analytics.events    ADD SEARCH OPTIMIZATION ON SUBSTRING(url);

-- These can now use the access path instead of a full scan.
SELECT customer_id, email FROM analytics.customers WHERE email = 'someone@example.com';
SELECT event_id FROM analytics.events WHERE url LIKE '%/checkout/complete%';

-- Inspect, then remove when it stops paying.
SHOW SEARCH OPTIMIZATION ON analytics.customers;
ALTER TABLE analytics.customers DROP SEARCH OPTIMIZATION ON EQUALITY(email);
```

## Materialized views

### Explanation
Dashboards often run the same expensive aggregation over a large table many times a day. A **materialized view**
stores the result of a query as a Snowflake-managed internal table and keeps it up to date for you, so repeated
reads are cheap.

Snowflake maintains the view incrementally from the base table's change tracking, using serverless compute. Because
maintenance must be incremental, the definition is restricted:

```text
allowed:      one base table, a subset of aggregates and scalar expressions
not allowed:  joins, HAVING, window functions, user-defined functions
```

The optimizer can also rewrite a query against the base table to read the materialized view when it can prove the
result is the same. So application SQL does not have to change to benefit. Think of it as a pre-cooked dish that the
kitchen keeps fresh and serves whenever someone orders the same thing.

`ALTER MATERIALIZED VIEW ... SUSPEND` stops maintenance. The view may then be stale until you resume it.

### Use cases
- **Daily revenue dashboard:** pre-aggregate `SUM(amount)` and `COUNT(*)` by `order_date` from a large orders table.
- **Repeated heavy rollups:** many BI users run the same single-table `GROUP BY` all day.
- **Maintenance pause:** suspend the view during a large backfill of the base table, then resume.

### Limitations and things to remember
- Single base table only: no joins, no `HAVING`, no window functions, no UDFs; only some aggregates are allowed.
- An automatic rewrite is not guaranteed. Predicates or expressions the optimizer cannot map to the view's grouping,
  or joins against the base table, may prevent it. Confirm the rewrite in the query profile.
- Maintenance bills serverless credits; a high-churn base table can cost more to maintain than the view saves.
- A suspended view may be stale until resumed.
- Track cost in `MATERIALIZED_VIEW_REFRESH_HISTORY`.

### Code example

The code creates a daily revenue materialized view, shows a query the optimizer can rewrite, and reads maintenance cost.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE MATERIALIZED VIEW analytics.mv_daily_revenue AS
SELECT order_date, SUM(amount) AS revenue, COUNT(*) AS orders
FROM analytics.orders
GROUP BY order_date;

-- The optimizer can rewrite this base-table query to read the view.
SELECT order_date, SUM(amount) FROM analytics.orders GROUP BY order_date;

ALTER MATERIALIZED VIEW analytics.mv_daily_revenue SUSPEND;   -- may become stale
ALTER MATERIALIZED VIEW analytics.mv_daily_revenue RESUME;

-- Serverless credits spent maintaining each view.
SELECT name, SUM(credits_used) AS credits
FROM snowflake.account_usage.materialized_view_refresh_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY name
ORDER BY credits DESC;
```

## Query acceleration service

### Explanation
Most queries on a warehouse are small, but an occasional one scans a huge table. Sizing the warehouse for that worst
query wastes credits the rest of the time. The **query acceleration service** solves this by lending the warehouse
extra serverless compute for eligible queries only.

For an eligible query, the service processes part of the scan and part of the aggregation on serverless compute,
while the warehouse handles the rest:

```text
large filtered scan + aggregation
        |-- warehouse (normal size) --> part of the work
        |-- serverless acceleration --> part of the scan and aggregation
```

Eligibility is decided at compile time from the query shape. Scans with filters and aggregations over large tables
qualify. Plans dominated by joins, sorts or very selective lookups generally do not.

You enable it per warehouse with `ENABLE_QUERY_ACCELERATION`. `QUERY_ACCELERATION_MAX_SCALE_FACTOR` caps how much
serverless compute can be added, and it is the main guardrail on spend. `SYSTEM$ESTIMATE_QUERY_ACCELERATION` tells
you whether a specific past query would benefit.

### Use cases
- **Mixed BI warehouse:** keep `bi_wh` small and let acceleration absorb the occasional large ad-hoc scan.
- **Unpredictable analyst queries:** big filtered aggregations over history tables that run a few times a week.
- **Pre-check:** run `SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>')` on a slow query before enabling it.

### Limitations and things to remember
- It is billed separately from the warehouse, so a query can become faster and more expensive at the same time.
- It cannot fix a full scan that pruning could have avoided; fix the predicate first.
- Joins, sorts and very selective lookups usually do not qualify.
- Credits appear in `QUERY_ACCELERATION_HISTORY`; per-query use appears as `QUERY_ACCELERATION_BYTES_SCANNED` and
  `QUERY_ACCELERATION_TIME` in `QUERY_HISTORY`.
- Set `QUERY_ACCELERATION_MAX_SCALE_FACTOR` deliberately to cap spend.

### Code example

The code enables query acceleration with a spend cap, estimates benefit for one query, and lists queries that used it.

Snowflake SQL (shown, not run in this notebook):

```sql
ALTER WAREHOUSE bi_wh SET
  ENABLE_QUERY_ACCELERATION = TRUE
  QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;   -- cap on added serverless compute

-- Would this specific query benefit?
SELECT SYSTEM$ESTIMATE_QUERY_ACCELERATION('<query_id>');

-- Which queries actually used acceleration?
SELECT query_id, query_acceleration_bytes_scanned, query_acceleration_time,
       total_elapsed_time / 1000 AS elapsed_s
FROM snowflake.account_usage.query_history
WHERE query_acceleration_bytes_scanned > 0
  AND start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
ORDER BY query_acceleration_bytes_scanned DESC
LIMIT 20;
```

## Time Travel and Fail-safe

### Explanation
People make mistakes: a bad `DELETE`, a wrong `UPDATE`, a dropped table. Time Travel lets you undo them with plain SQL.

Snowflake never edits data in place. Every DML statement writes new micro-partitions (a micro-partition is a 50-500 MB block of rows that Snowflake stores and manages for you) and keeps the old ones. The **retention period** is how many days Snowflake keeps those old versions. While they exist, you can query the table as it was in the past, restore a dropped object with `UNDROP`, or clone an old version.

`AT` and `BEFORE` accept a timestamp, an offset in seconds, or a statement (query) id. `AT (STATEMENT => id)` includes that statement's effects; `BEFORE (STATEMENT => id)` shows the state just before it ran.

**Fail-safe** is a second, fixed seven-day window that starts when Time Travel ends. You cannot query it. Only Snowflake support can recover data from it.

```text
change ──► [ Time Travel: 0-90 days, you query it ] ──► [ Fail-safe: 7 days, support only ] ──► gone
```

Think of Time Travel as your own undo button and Fail-safe as the vendor's last-resort backup. Neither replaces replication and exports for real disaster recovery.

### Use cases
- **Accidental delete:** restore a table from just before the bad statement with `BEFORE (STATEMENT => ...)`.
- **Dropped object:** bring back a table, schema, or database with `UNDROP` while retention is open.
- **Audit snapshot:** clone a table `AT` a timestamp to keep a durable point-in-time copy after retention passes.
- **Debugging a pipeline:** compare row counts now versus an hour ago with `AT (OFFSET => -3600)`.

### Limitations and things to remember
- Retention is set at database, schema, or table level with `DATA_RETENTION_TIME_IN_DAYS`.
- Standard Edition allows 0 or 1 day; Enterprise Edition allows 0 to 90 days.
- Fail-safe is a fixed 7 days, applies to permanent tables only, and is not queryable.
- Transient and temporary tables get at most 1 day of Time Travel and no Fail-safe.
- Longer retention costs storage: superseded micro-partitions stay stored for the whole window, so a heavily changing table with 90-day retention pays for all that history.
- Use transient tables for rebuildable staging, where Fail-safe adds no value.

### Code example
The code sets retention, queries the past in three ways, and recovers data.

Snowflake SQL (shown, not run in this notebook):

```sql
ALTER TABLE analytics.orders SET DATA_RETENTION_TIME_IN_DAYS = 30;

-- Query the past: absolute time, offset in seconds, or just before a statement.
SELECT COUNT(*) FROM analytics.orders AT (TIMESTAMP => '2026-10-01 00:00:00'::TIMESTAMP);
SELECT COUNT(*) FROM analytics.orders AT (OFFSET => -3600);
SELECT COUNT(*) FROM analytics.orders BEFORE (STATEMENT => '<query_id>');

-- Recover a dropped table, or keep a point-in-time copy without copying data.
UNDROP TABLE analytics.orders;
CREATE TABLE analytics.orders_audit CLONE analytics.orders AT (OFFSET => -86400);

-- Check how much history the table keeps.
SELECT table_name, retention_time, dropped, created
FROM analytics.information_schema.tables
WHERE table_name = 'ORDERS';
```

The runnable cell below simulates Time Travel with DuckDB: each DML statement saves a table version, and we restore the table from just before an accidental `DELETE`.

In [ ]:
import duckdb

con = duckdb.connect()

# Every DML statement produces a new table version; old versions are kept, not overwritten.
versions: list[tuple[str, str]] = []  # (statement_id, snapshot table name)


def run_dml(statement_id: str, sql: str) -> None:
    con.execute(sql)
    snapshot = f"orders_v{len(versions)}"
    con.execute(f"CREATE TABLE {snapshot} AS SELECT * FROM orders")  # Snowflake shares partitions instead
    versions.append((statement_id, snapshot))


def before(statement_id: str) -> str:
    """Return the snapshot for: SELECT ... FROM orders BEFORE (STATEMENT => statement_id)."""
    position = [s for s, _ in versions].index(statement_id)
    return versions[position - 1][1]


con.execute("CREATE TABLE orders (order_id INTEGER, status VARCHAR)")
run_dml("q1", "INSERT INTO orders VALUES (1, 'created'), (2, 'created'), (3, 'shipped')")
run_dml("q2", "UPDATE orders SET status = 'shipped' WHERE order_id = 1")
run_dml("q3", "DELETE FROM orders")  # the accident

print("now:", con.execute("SELECT count(*) FROM orders").fetchone()[0], "rows")
# Snowflake: CREATE TABLE orders_restored CLONE orders BEFORE (STATEMENT => 'q3')
con.execute(f"CREATE TABLE orders_restored AS SELECT * FROM {before('q3')}")
restored = con.execute("SELECT * FROM orders_restored ORDER BY order_id").fetchall()
print("restored:", restored)
assert restored == [(1, "shipped"), (2, "created"), (3, "shipped")]

## Zero-copy cloning

### Explanation
Teams need production-sized copies for development, testing, and audits. A real copy of a large database is slow and doubles storage. Zero-copy cloning solves this.

A **clone** is a new database, schema, or table that copies only the source's metadata. It records a pointer to each of the source's micro-partitions instead of copying them. Both objects read the same files.

Micro-partitions are immutable. When either side changes rows, Snowflake writes new partitions for the affected rows and leaves the shared ones alone. This is **copy-on-write** at partition granularity, not row granularity. The two objects diverge only where they differ.

```text
after CLONE:      source ─┬─► P1 P2 P3 P4      (shared, no new storage)
                  clone  ─┘
after UPDATE on clone:
                  source ───► P1 P2 P3 P4
                  clone  ───► P1 P2 P3 P4'     (only P4' is new storage)
```

So a 100 TB database clones in seconds and adds almost no storage. If a test rewrites 5 percent of the clone's partitions, it adds roughly 5 percent of the data, not 100 percent. A clone modified everywhere approaches the cost of a full copy.

It is like a shared document where each editor's changes are saved as new pages, while untouched pages stay shared.

### Use cases
- **Dev sandbox:** clone the whole `analytics` database so engineers test against production-sized data.
- **CI test runs:** clone, run dbt models against the clone, then drop it.
- **Audit snapshot:** clone a table `AT` a timestamp to freeze month-end numbers.
- **Risky migration:** clone before a large backfill so you can compare or roll back.

### Limitations and things to remember
- Privileges do not travel with the clone; re-apply grants explicitly.
- Storage is shared, so dropping the source while a clone exists frees nothing.
- A clone starts its own Time Travel history at creation; it does not inherit the source's history.
- To capture a later retention window, clone again.
- Cost grows with the share of partitions the clone (or source) rewrites.

### Code example
The code clones a database and a point-in-time table, modifies the clone, and re-applies grants.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Clone an entire database for a safe development sandbox (seconds, no data copied).
CREATE DATABASE analytics_dev CLONE analytics;

-- Snapshot one table at a point in time.
CREATE TABLE analytics.orders_snapshot CLONE analytics.orders
  AT (TIMESTAMP => '2026-10-01 00:00:00'::TIMESTAMP);

-- Changing the clone writes new partitions; production is untouched.
UPDATE analytics_dev.marts.daily_revenue SET revenue = 0 WHERE order_date < '2026-01-01';

-- Grants do not travel with the clone.
GRANT USAGE ON DATABASE analytics_dev TO ROLE analytics_engineer;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics_dev.marts TO ROLE analytics_engineer;

DROP DATABASE analytics_dev;
```

## Stages and COPY INTO

### Explanation
Data usually lands as files in cloud storage. You need a safe way to point Snowflake at those files and load them fast, without loading the same file twice.

A **stage** is a named pointer to a storage location. An external stage points at your own bucket, such as S3. A **storage integration** lets the stage reach the bucket without embedding credentials.

`COPY INTO <table>` loads files from a stage in parallel. `COPY INTO @stage` does the reverse and unloads query results into files.

**Load metadata** is Snowflake's memory of which files it has already loaded into a table. It lasts 64 days. With `FORCE = FALSE` Snowflake checks it and skips files already loaded; `FORCE = TRUE` ignores it. The memory belongs to the table, so loading the same file into a second table loads it again.

`ON_ERROR` sets the error policy per statement:

```text
ABORT_STATEMENT  -> fail the whole load
CONTINUE         -> load good rows, record bad ones
SKIP_FILE        -> skip any file that contains an error
```

File size drives parallelism. Compressed files of about 100 to 250 MB keep many files in flight across the warehouse. One multi-gigabyte file limits how much of the warehouse can work.

### Use cases
- **Daily batch load:** load one day of Parquet files with `PATTERN` and `PURGE = TRUE`.
- **Pre-flight check:** run `VALIDATION_MODE = 'RETURN_ERRORS'` to find bad rows before loading.
- **Data export:** unload a mart to Parquet files for a partner or another engine.
- **Load triage:** query `COPY_HISTORY` when expected data never appears.

### Limitations and things to remember
- Load metadata lasts 64 days and is per table.
- Aim for 100-250 MB compressed files; avoid one huge file.
- For unloads, `MAX_FILE_SIZE` splits output into many files for the same parallelism reason.
- For Snowpipe, the `ON_ERROR` policy lives in the pipe definition.
- Every outcome lands in `COPY_HISTORY` with an `error_count` and a status; check it first when data is missing.

### Code example
The code creates a credential-free external stage, loads and validates files, unloads results, and checks load history.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Credential-free access to your bucket.
CREATE OR REPLACE STORAGE INTEGRATION s3_orders_int
  TYPE = EXTERNAL_STAGE  STORAGE_PROVIDER = 'S3'  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('s3://acme-raw/orders/');

CREATE OR REPLACE STAGE raw.orders.landing
  URL = 's3://acme-raw/orders/'
  STORAGE_INTEGRATION = s3_orders_int
  FILE_FORMAT = (TYPE = PARQUET);

-- Load one day of files, keep good rows, delete files after success, skip already-loaded files.
COPY INTO raw.orders.orders
FROM @raw.orders.landing
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
PATTERN = '.*2026-10-04.*[.]parquet'
ON_ERROR = 'CONTINUE'
PURGE = TRUE
FORCE = FALSE;

-- Dry run: report errors without loading.
COPY INTO raw.orders.orders FROM @raw.orders.landing
  FILE_FORMAT = (TYPE = PARQUET) VALIDATION_MODE = 'RETURN_ERRORS';

-- Unload a result set into ~100 MB files.
COPY INTO @raw.orders.exports/daily/
FROM (SELECT * FROM analytics.marts.daily_revenue WHERE order_date = '2026-10-04')
FILE_FORMAT = (TYPE = PARQUET) MAX_FILE_SIZE = 100000000 HEADER = TRUE;

-- Load history for the last day, including errors.
SELECT file_name, status, row_count, error_count, first_error_message, last_load_time
FROM TABLE(information_schema.copy_history(
  table_name => 'raw.orders.orders',
  start_time => DATEADD('day', -1, CURRENT_TIMESTAMP())))
ORDER BY last_load_time DESC;
```

## Snowpipe and Snowpipe Streaming

### Explanation
Scheduled `COPY INTO` jobs leave data waiting until the next run. When files arrive all day, you want each file loaded soon after it lands, without running a warehouse around the clock.

A **pipe** is a stored `COPY INTO` statement plus its own load history. With `AUTO_INGEST = TRUE`, Snowflake gives you a notification channel (an SQS queue, a Pub/Sub subscription, or an Event Grid endpoint). You register it with the bucket. Each object-created event then triggers the pipe, which loads just that file. Load metadata records every file, so a redelivered event does not load it twice.

```text
file lands in bucket ──► cloud event ──► pipe (COPY INTO) ──► table
                                          serverless credits
```

Snowpipe runs on Snowflake-managed compute billed as serverless credits. Every file carries its own event, metadata, and processing overhead.

**Snowpipe Streaming** skips files entirely. A client writes rows through the API and Snowflake commits them into the table. It trades per-file overhead for row-level latency.

Rule of thumb: scheduled `COPY INTO` for large, predictable batches; Snowpipe for continuous file arrival; Snowpipe Streaming for row-level latency.

### Use cases
- **Event files from apps:** load small Parquet drops into a raw table within minutes of arrival.
- **Partner uploads:** load files whenever a partner pushes them, with no fixed schedule.
- **Kafka or CDC feeds:** use Snowpipe Streaming to commit rows directly with low latency.
- **Cost review:** rank pipes by serverless credits in `PIPE_USAGE_HISTORY`.

### Limitations and things to remember
- Delivery is at-least-once: a file or row can be processed twice.
- Make the target or the downstream merge key on a business key or file name; the pipe will not make duplicates safe.
- Thousands of tiny files multiply both latency and cost.
- `AUTO_INGEST` needs the notification channel registered with the bucket.
- Check `SYSTEM$PIPE_STATUS` and `COPY_HISTORY` when files do not appear.

### Code example
The code creates an auto-ingest pipe, then checks its status, recent loads, and credit use.

Snowflake SQL (shown, not run in this notebook):

```sql
-- File-based continuous ingestion driven by cloud event notifications.
CREATE OR REPLACE PIPE raw.orders.orders_pipe AUTO_INGEST = TRUE
AS COPY INTO raw.orders.orders
   FROM @raw.orders.orders_landing
   FILE_FORMAT = (TYPE = PARQUET)
   MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE;

-- Pipe state; SHOW PIPES also lists the notification channel to register with the bucket.
SELECT SYSTEM$PIPE_STATUS('raw.orders.orders_pipe');
SHOW PIPES LIKE 'ORDERS_PIPE' IN SCHEMA raw.orders;

-- What the pipe loaded in the last 24 hours, and what failed.
SELECT file_name, row_count, error_count, status, last_load_time
FROM TABLE(information_schema.copy_history(
  table_name => 'raw.orders.orders',
  start_time => DATEADD('hour', -24, CURRENT_TIMESTAMP())));

-- Serverless credits per pipe over the last week.
SELECT pipe_name, SUM(credits_used) AS credits
FROM snowflake.account_usage.pipe_usage_history
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY pipe_name
ORDER BY credits DESC;
```

## Semi-structured data with VARIANT

### Explanation
Events and API responses arrive as JSON whose shape changes over time. Forcing them into fixed columns at load time breaks pipelines when a field appears or moves. `VARIANT` solves this.

**`VARIANT`** is a column type that holds any JSON-like value: objects, arrays, numbers, or strings. You can load raw documents as they are and decide later which fields matter.

On load, Snowflake parses each document and stores every distinct path in an internal columnar form. A key repeated across millions of rows is stored once as a path, with its values beside it, not as text in every row. Variant paths have statistics, but weaker ones than typed columns. So a filter on `payload:user.id` prunes worse than one on a typed `user_id`.

Path syntax:

```text
payload:user.id        object key (nested)
payload:items[0]       array element
payload:"user id"      key with a space (or payload['user id'])
payload:amount::NUMBER cast to a type
```

**`FLATTEN`** turns an array into rows and exposes `VALUE`, `PATH`, `INDEX`, and `THIS`. **`TRY_CAST`** converts a value and returns null instead of failing.

The pattern: keep raw `VARIANT` for replay and schema drift, explore it with `:`, `TRY_CAST`, and `FLATTEN`, then promote the fields people query into typed columns.

### Use cases
- **Clickstream ingestion:** land raw JSON events in a `VARIANT` column, so new fields never break the load.
- **Order line items:** explode a nested `items` array into one row per SKU with `LATERAL FLATTEN`.
- **Schema exploration:** use `TYPEOF` and `OBJECT_KEYS` to see what a new feed actually contains.
- **Typed serving view:** expose hot fields as typed columns for BI joins and pruning.

### Limitations and things to remember
- Pruning on variant paths is weaker than on typed columns.
- `TRY_CAST` returns null both when a value is missing and when it is malformed, so a quiet null can hide a schema change.
- Validate business-critical paths with `TYPEOF` or `OBJECT_KEYS`.
- Promote frequently used paths into typed columns instead of recasting them in every query.
- Keys with spaces or special characters need double quotes or bracket syntax.
- Both `:` and `[]` accept a constant or a computed key.

### Code example
The code loads raw JSON into a `VARIANT` column, explores it, extracts typed fields, flattens an array, and promotes fields into a view.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE TABLE raw.events (event_id STRING, payload VARIANT);

-- Load whole documents, pulling out one key as a column.
COPY INTO raw.events (event_id, payload)
FROM (SELECT $1:event_id::STRING, $1 FROM @raw.events.event_stage)
FILE_FORMAT = (TYPE = JSON STRIP_OUTER_ARRAY = TRUE);

-- Explore before you flatten.
SELECT payload:event_type::STRING AS event_type, TYPEOF(payload:amount) AS amount_type,
       OBJECT_KEYS(payload) AS keys
FROM raw.events LIMIT 5;

-- Typed fields with a safe cast.
SELECT payload:user.id::NUMBER AS user_id,
       TRY_CAST(payload:amount AS NUMBER(12, 2)) AS amount
FROM raw.events
WHERE payload:event_type::STRING = 'purchase';

-- Explode an array into rows.
SELECT e.event_id, i.value:sku::STRING AS sku, i.value:qty::NUMBER AS qty
FROM raw.events e, LATERAL FLATTEN(input => e.payload:items) i;

-- Promote the fields people actually use.
CREATE OR REPLACE VIEW analytics.v_events AS
SELECT event_id,
       payload:event_type::STRING AS event_type,
       payload:user.id::NUMBER AS user_id,
       TRY_CAST(payload:amount AS NUMBER(12, 2)) AS amount,
       payload:ts::TIMESTAMP_NTZ AS event_ts
FROM raw.events;
```

The runnable cell below simulates `VARIANT` path access, `TRY_CAST`, and `LATERAL FLATTEN` with DuckDB's JSON type.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE raw_events AS SELECT event_id, payload::JSON AS payload FROM (VALUES
        ('e1', '{"event_type": "purchase", "user": {"id": 42}, "amount": "19.90",
                 "items": [{"sku": "A", "qty": 2}, {"sku": "B", "qty": 1}]}'),
        ('e2', '{"event_type": "purchase", "user": {"id": 7}, "amount": "n/a", "items": []}'),
        ('e3', '{"event_type": "view", "user": {"id": 42}}')
    ) t(event_id, payload)
""")

# Snowflake: payload:user.id::NUMBER and TRY_CAST(payload:amount AS NUMBER(12, 2))
typed = con.execute("""
    SELECT event_id, CAST(payload ->> '$.user.id' AS INTEGER) AS user_id,
           TRY_CAST(payload ->> '$.amount' AS DECIMAL(12, 2)) AS amount
    FROM raw_events WHERE payload ->> '$.event_type' = 'purchase' ORDER BY event_id
""").fetchall()

# Snowflake: LATERAL FLATTEN(input => payload:items)
items = con.execute("""
    SELECT e.event_id, i ->> '$.sku' AS sku, CAST(i ->> '$.qty' AS INTEGER) AS qty
    FROM raw_events e, UNNEST(CAST(payload -> '$.items' AS JSON[])) AS u(i)
    ORDER BY sku
""").fetchall()

print(typed)
print(items)
assert typed[1][2] is None  # TRY_CAST: a malformed value becomes a quiet NULL
assert items == [("e1", "A", 2), ("e1", "B", 1)]

## External tables

### Explanation
Not all data is worth loading. Years of cold history in a data lake cost time and storage to copy into Snowflake. External tables let you query those files where they are.

An **external table** stores no row data in Snowflake. It holds only column definitions and a catalog of the files and partitions found in a stage. At query time the warehouse opens the files directly. It picks files using **partition columns** derived from the file path and the statistics in each file's footer.

So pruning (skipping files that cannot match a filter) depends on your directory layout and the file format's own statistics, not on Snowflake micro-partitions.

```text
s3://lake/events/event_date=2026-10-04/part-001.parquet   <- date filter can skip other folders
s3://lake/events/all/part-001.parquet                     <- every query opens every file
```

The file catalog must stay current. `AUTO_REFRESH = TRUE` adds new files when the stage has event notifications. Without them, the list stays stale until you run `ALTER EXTERNAL TABLE ... REFRESH`.

A good split: keep hot, frequently joined data in native tables and leave cold history external.

### Use cases
- **Cold history:** query years of archived events in the lake without loading them.
- **Exploring a new feed:** inspect lake files before deciding to build a load pipeline.
- **Lineage checks:** use `METADATA$FILENAME` and `METADATA$FILE_ROW_NUMBER` to trace rows to source files.
- **Shared lake data:** read files that other engines also produce, with no copy.

### Limitations and things to remember
- Read-only: no DML writes.
- No Time Travel and no clustering.
- `AUTO_REFRESH` works only when the stage has event notifications; otherwise refresh manually.
- Putting every date in one directory removes the only cheap pruning signal.
- Name prefixes by partition, such as `event_date=2026-10-04/`, so date filters skip unrelated files.

### Code example
The code defines an external table over Parquet files, refreshes its file list, and queries it with file metadata.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE EXTERNAL TABLE raw.lake.events_ext (
  event_id STRING AS (VALUE:c1::STRING),
  payload  VARIANT AS (VALUE)
)
LOCATION = @raw.lake.events_stage/events/
FILE_FORMAT = (TYPE = PARQUET)
AUTO_REFRESH = TRUE;

-- Manual refresh when the stage has no event notifications.
ALTER EXTERNAL TABLE raw.lake.events_ext REFRESH;

-- Query in place, with file metadata for lineage.
SELECT METADATA$FILENAME, METADATA$FILE_ROW_NUMBER, event_id, payload:event_type::STRING
FROM raw.lake.events_ext
WHERE event_id IS NOT NULL
LIMIT 10;

-- Count files seen, useful when pruning looks wrong.
SELECT COUNT(*) AS files, MIN(METADATA$FILENAME) AS sample_file
FROM raw.lake.events_ext;
```

## Iceberg tables

### Explanation
Many companies want Snowflake, Spark, and Trino to work on the same data without copying it between systems. Apache Iceberg is an open table format that makes this possible.

An **Iceberg table** is a set of Parquet data files plus a metadata tree:

```text
snapshot ──► manifest list ──► manifests ──► Parquet data files
```

Manifests list data files, a manifest list lists manifests, and a snapshot points at one manifest list. A writer commits by adding a new snapshot atomically. So several engines can share a table without coordinating on files, as long as only one writer owns a table at a time.

Snowflake reaches the storage through an **external volume**, a named connection to your bucket. It needs one even for catalog-linked tables. There are two kinds of Iceberg table:

- **Snowflake-managed** (`CATALOG = 'SNOWFLAKE'`): Snowflake writes the data, maintains the metadata, and can apply Snowflake features.
- **Catalog-linked:** an external catalog (for example AWS Glue) owns the metadata. Snowflake is generally a reader.

Openness moves responsibility to you. File compaction, snapshot expiration, and storage lifecycle are no longer handled by a warehouse-native format. Decide up front which engine writes each table.

### Use cases
- **Shared lakehouse table:** Spark writes, Snowflake and Trino read the same Iceberg table.
- **Avoiding lock-in:** keep core tables in an open format that any engine can read.
- **Snowflake as the writer:** build marts in Snowflake-managed Iceberg tables that other engines can query.
- **Existing Glue catalog:** register tables already managed in Glue and query them from Snowflake.

### Limitations and things to remember
- An external volume is required, even for catalog-linked tables.
- Cross-engine writers can commit conflicting snapshots; designate a single writer per table.
- You own compaction, snapshot expiration, and storage lifecycle.
- In catalog-linked tables, the external catalog owns the metadata and Snowflake is generally a reader.

### Code example
The code creates an external volume, a Snowflake-managed Iceberg table, and a catalog-linked one, then writes and reads data.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE EXTERNAL VOLUME iceberg_vol
  STORAGE_LOCATIONS = ((
    NAME = 's3_primary'
    STORAGE_PROVIDER = 'S3'
    STORAGE_BASE_URL = 's3://acme-iceberg/'
    STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::111122223333:role/snowflake-iceberg'
  ));

-- Snowflake-managed: Snowflake writes and maintains the Iceberg metadata.
CREATE OR REPLACE ICEBERG TABLE analytics.orders_iceberg (
  order_id NUMBER, order_date DATE, amount NUMBER(12, 2)
)
CATALOG = 'SNOWFLAKE'
EXTERNAL_VOLUME = 'iceberg_vol'
BASE_LOCATION = 'analytics/orders';

-- Catalog-linked: an external catalog (here Glue) owns the table.
CREATE OR REPLACE ICEBERG TABLE analytics.orders_glue
  CATALOG = 'glue_catalog_integration'
  CATALOG_TABLE_NAME = 'orders'
  EXTERNAL_VOLUME = 'iceberg_vol';

INSERT INTO analytics.orders_iceberg
SELECT order_id, order_date, amount FROM raw.orders_staging;

SELECT COUNT(*) AS rows, COUNT(DISTINCT order_date) AS days FROM analytics.orders_iceberg;
```

## Streams (change data capture)

### Explanation
Change data capture (CDC) means picking up only the rows that changed since the last load, instead of re-reading a whole table. In Snowflake you do this with a stream. A stream is a change cursor on a table: it does not copy rows, it stores an offset (a bookmark) into the table's change history plus a state flag.

A plain `SELECT` from a stream shows the pending changes and never moves the offset, no matter how often it runs. The offset moves only when a DML statement (`INSERT`, `MERGE`, and so on) reads the stream and writes the result. The move happens inside the same transaction as the write, so a rollback leaves the stream pointing at the same changes. Think of a bookmark that moves only when you finish copying the page.

```text
source table changes:  c1  c2  c3  c4
                        ^offset           SELECT: shows c1..c4, offset stays
MERGE ... FROM stream  ------------->     COMMIT: offset jumps past c4
```

There are two kinds. A standard stream captures inserts, updates and deletes and adds the columns `METADATA$ACTION`, `METADATA$ISUPDATE` and `METADATA$ROW_ID`. An append-only stream captures inserts only, does less bookkeeping, and is cheaper for immutable event tables. Choose append-only unless you really need update and delete tracking. Because one key can change several times in one delta, the consumer must merge idempotently, which means running it twice gives the same result.

### Use cases
- **Incremental upsert to a mart:** merge only changed orders into `analytics.orders` instead of rebuilding it.
- **Event table ingestion:** an append-only stream on a raw clickstream table feeds downstream loads cheaply.
- **Gating a scheduled job:** check `SYSTEM$STREAM_HAS_DATA` so a task skips runs when nothing changed.
- **Propagating deletes:** a standard stream carries `DELETE` actions so the target drops removed rows.

### Limitations and things to remember
- Only DML that reads the stream advances it; `SELECT` alone never does.
- The offset advances atomically with the write; a rollback leaves the delta in place to retry.
- A stream goes stale when the source's data retention window passes a point it still needs.
- `MAX_DATA_EXTENSION_TIME_IN_DAYS` (14 by default) extends retention for unconsumed streams.
- A stale stream cannot be recovered: drop and recreate it, losing every change in between.
- The same key can appear several times in one delta, so deduplicate (for example with `QUALIFY ROW_NUMBER()`) before merging.

### Code example

The code creates a stream, peeks at it without consuming, and consumes it with an idempotent `MERGE`.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE STREAM raw.orders.orders_stream ON TABLE raw.orders.orders;

-- Peek at the pending delta: this does NOT advance the offset.
SELECT METADATA$ACTION, METADATA$ISUPDATE, order_id, status
FROM raw.orders.orders_stream LIMIT 10;

-- Consuming in DML advances the offset atomically with the write.
MERGE INTO analytics.orders AS t
USING (
  SELECT order_id, status, updated_at, METADATA$ACTION AS action
  FROM raw.orders.orders_stream
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) = 1
) AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED AND s.action = 'INSERT'
  THEN INSERT (order_id, status, updated_at) VALUES (s.order_id, s.status, s.updated_at);

-- Is there anything to do?
SELECT SYSTEM$STREAM_HAS_DATA('raw.orders.orders_stream');
```

The runnable cell below simulates a stream with DuckDB: a change log, an offset that reads never move, and a `MERGE` that advances the offset only when its transaction commits.

In [ ]:
import duckdb


class Stream:
    """A Snowflake-style stream: an offset into the source table's change log."""

    def __init__(self, con, change_log: str):
        self.con, self.change_log, self.offset = con, change_log, 0

    def pending(self) -> list[tuple]:  # SELECT * FROM stream: never advances the offset
        return self.con.execute(f"SELECT * FROM {self.change_log} WHERE seq > ? ORDER BY seq",
                                [self.offset]).fetchall()

    def consume_into(self, merge_sql: str) -> None:  # DML that reads the stream advances it atomically
        high = self.con.execute(f"SELECT coalesce(max(seq), 0) FROM {self.change_log}").fetchone()[0]
        self.con.execute("BEGIN")
        try:
            self.con.execute(merge_sql, [self.offset, high])
            self.con.execute("COMMIT")
            self.offset = high
        except Exception:
            self.con.execute("ROLLBACK")  # offset unchanged: the delta is retried, not lost
            raise


con = duckdb.connect()
con.execute("""
    CREATE TABLE orders_changes (seq INTEGER, action VARCHAR, order_id INTEGER, status VARCHAR, updated_at INTEGER);
    INSERT INTO orders_changes VALUES
        (1, 'INSERT', 10, 'created', 100), (2, 'INSERT', 11, 'created', 101),
        (3, 'INSERT', 10, 'shipped', 105),  -- same key changed twice in one batch
        (4, 'DELETE', 11, NULL, 106);
    CREATE TABLE analytics_orders (order_id INTEGER, status VARCHAR, updated_at INTEGER);
""")
MERGE_FROM_STREAM = """
    MERGE INTO analytics_orders t
    USING (SELECT * FROM orders_changes WHERE seq > ? AND seq <= ?
           QUALIFY row_number() OVER (PARTITION BY order_id ORDER BY seq DESC) = 1) s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
    WHEN MATCHED THEN UPDATE SET status = s.status, updated_at = s.updated_at
    WHEN NOT MATCHED AND s.action = 'INSERT' THEN INSERT VALUES (s.order_id, s.status, s.updated_at)
"""
stream = Stream(con, "orders_changes")
assert len(stream.pending()) == len(stream.pending()) == 4  # reading twice does not consume
stream.consume_into(MERGE_FROM_STREAM)
print(con.execute("SELECT * FROM analytics_orders").fetchall(), "| pending:", stream.pending())
assert con.execute("SELECT * FROM analytics_orders").fetchall() == [(10, "shipped", 105)]
assert stream.pending() == []

## Tasks

### Explanation
A pipeline needs something to run its steps on a schedule and in the right order. In Snowflake that is a task. A task is a named SQL statement (often a `CALL` to a procedure) that Snowflake runs for you. Tasks link into a DAG (directed acyclic graph): a chain of steps where each step runs after the ones it depends on.

A root or standalone task has a `SCHEDULE`, either a plain interval or a cron expression with a time zone. A child task has `AFTER <parent>` and no schedule. When the root runs, children run as their predecessors succeed. Add `WHEN SYSTEM$STREAM_HAS_DATA(...)` to the root and the task skips runs when its stream is empty, so you get an incremental pipeline inside Snowflake.

```text
load_task (every 5 min, WHEN stream has data)
   └── transform_task (AFTER load_task)
```

Tasks run on one of two kinds of compute. A warehouse-backed task names a `WAREHOUSE` and uses its credits. A serverless task omits `WAREHOUSE` and runs on Snowflake-managed compute, billed as serverless credits in `SERVERLESS_TASK_HISTORY`.

A task graph is not one transaction. If a child fails, its committed parents stay applied, and a retry re-runs only from the failed point. So every task body must be idempotent.

### Use cases
- **Stream-driven incremental load:** a root task every 5 minutes merges a stream only when it has data.
- **Load-then-transform chain:** a child task builds marts after the load task succeeds.
- **Nightly housekeeping:** a serverless cron task runs compaction or audit inserts at 3 AM local time.
- **Manual backfill run:** `EXECUTE TASK` triggers the root once, outside the schedule.

### Limitations and things to remember
- Resume child tasks first, then the root; new tasks start suspended.
- Graphs are versioned: a run finishes against the definition that existed when it started, so mid-run edits do not change it.
- `EXECUTE TASK` runs a standalone task or a graph root even when suspended, but does not resume suspended children.
- A failed subtree does not roll back its parents; make every step idempotent.
- Handle overlap on purpose so a slow run and the next scheduled run do not fight over the same rows.
- A `WHEN SYSTEM$STREAM_HAS_DATA` gate means an idle schedule costs nothing on a warehouse-backed task.

### Code example

The code builds a two-step graph gated on a stream, adds a serverless cron task, and reads run history.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE TASK raw.orders.load_task
  WAREHOUSE = elt_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw.orders.orders_stream')
AS CALL analytics.load_orders_incremental();

CREATE OR REPLACE TASK analytics.transform_task
  WAREHOUSE = transform_wh
  AFTER raw.orders.load_task
AS CALL analytics.build_orders_marts();

-- Resume children first, then the root.
ALTER TASK analytics.transform_task RESUME;
ALTER TASK raw.orders.load_task RESUME;

-- Serverless: no WAREHOUSE, cron with a time zone.
CREATE OR REPLACE TASK analytics.compact_task
  SCHEDULE = 'USING CRON 0 3 * * * America/Chicago'
AS INSERT INTO analytics.audit_log SELECT CURRENT_TIMESTAMP(), 'compacted';

-- Runs, states and errors over the last day.
SELECT name, state, scheduled_time, error_message
FROM TABLE(information_schema.task_history(
  scheduled_time_range_start => DATEADD('day', -1, CURRENT_TIMESTAMP())))
ORDER BY scheduled_time DESC;
```

## Dynamic tables

### Explanation
Streams plus tasks work, but every new consumer means another task to create, wire up and resume in the right order. A dynamic table removes that work. A dynamic table is a table defined by a query: you declare the result you want and a `TARGET_LAG`, the maximum staleness you accept, and Snowflake schedules the refreshes. It stores its own data and keeps a refresh history.

The refresh mode decides how Snowflake updates it. The default `REFRESH_MODE = AUTO` picks an incremental refresh (process only changes) when the query supports it, and a full refresh (recompute the whole result) when it does not. You can force `INCREMENTAL` or `FULL`.

Dynamic tables that read from each other form a dependency graph that Snowflake manages. Adding a new downstream table does not require editing upstream schedules. Setting `TARGET_LAG = DOWNSTREAM` means a table makes no freshness promise of its own; Snowflake refreshes it just often enough to keep its consumers on time.

```text
orders + customers --> daily_revenue (DOWNSTREAM) --> segment_revenue (lag 10 min)
```

Think of it as a materialized result with a thermostat: you set the target, Snowflake decides when to run. The catch is cost, which is hidden behind that automation.

### Use cases
- **SQL-shaped marts:** keep a daily revenue table within 10 minutes of its sources without writing tasks.
- **Layered transformations:** chain staging, intermediate and mart tables with `DOWNSTREAM` lag on the middle layers.
- **Replacing stream-plus-task plumbing:** add a new consumer without touching upstream schedules.

### Limitations and things to remember
- `TARGET_LAG` is a target, not a guarantee.
- A refresh slower than the lag, or a query that cannot be incrementalized, leaves the table reliably behind.
- A full refresh recomputes the whole result, which can be expensive on large inputs.
- Watch `DYNAMIC_TABLE_REFRESH_HISTORY` for refresh duration and state.
- Compare the achieved data timestamp with the requested lag; do not assume the lag is met.

### Code example

The code defines a dynamic table that feeds a downstream one, then checks state and refresh history.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE DYNAMIC TABLE analytics.daily_revenue
  TARGET_LAG = DOWNSTREAM          -- refresh only as often as consumers need
  WAREHOUSE = transform_wh
AS
  SELECT o.order_date, c.segment, SUM(o.amount) AS revenue, COUNT(*) AS orders
  FROM analytics.orders o JOIN analytics.customers c USING (customer_id)
  GROUP BY o.order_date, c.segment;

CREATE OR REPLACE DYNAMIC TABLE analytics.segment_revenue
  TARGET_LAG = '10 minutes'
  WAREHOUSE = transform_wh
AS
  SELECT segment, SUM(revenue) AS revenue
  FROM analytics.daily_revenue GROUP BY segment;

-- Requested lag versus actual state.
SELECT name, target_lag_sec, scheduling_state, latest_data_timestamp
FROM TABLE(information_schema.dynamic_tables());

-- Refresh duration and state per run.
SELECT name, state, refresh_start_time, refresh_end_time, data_timestamp
FROM TABLE(information_schema.dynamic_table_refresh_history(
  name => 'analytics.daily_revenue'));
```

## Snowpark

### Explanation
Many data engineers prefer to write transformations in Python rather than long SQL strings. Snowpark is Snowflake's library that lets you do this while still running the work next to the data. You build a DataFrame (a table-like object with methods such as `group_by` and `agg`), and nothing runs until you ask for a result. That is a lazy plan.

When you trigger it, the client sends the logical plan to cloud services, which compiles it to SQL and hands it to the warehouse. So a DataFrame expression does not run Python on the warehouse at all; it performs like the SQL it compiles to. Only code inside a user-defined function (UDF) or a stored procedure actually runs in a Python runtime.

```text
Python DataFrame --> logical plan --> SQL (cloud services) --> warehouse
UDF / procedure body --------------------------------------> Python runtime
```

UDFs come in two shapes. A scalar Python UDF is called once per row and pays interpreter overhead on every call. A vectorized pandas UDF receives batches as pandas Series and returns a Series, spreading that overhead across many rows. Use it when the logic cannot be written in native SQL.

A Snowpark-optimized warehouse has more memory and local SSD per node for memory-heavy UDFs and model scoring. It costs more per credit than a standard warehouse.

### Use cases
- **Python-first transformations:** build a daily revenue mart with DataFrame calls that push down as SQL.
- **Model scoring:** run a vectorized pandas UDF over a feature table on a Snowpark-optimized warehouse.
- **Custom parsing:** apply Python logic that has no native SQL function, in batches.

### Limitations and things to remember
- DataFrame transforms compile to SQL; only UDF and procedure code runs in Python.
- A scalar Python UDF over the same rows is much slower than the equivalent native function.
- Prefer native functions first, vectorized pandas UDFs second, scalar UDFs last.
- Snowpark-optimized warehouses cost more per credit; use them only when a job spills or truly needs the memory.
- Load connection credentials from a secrets store, not from code.

### Code example

Snowpark code is Python. It builds a DataFrame plan on your machine, and Snowflake runs it as SQL inside a warehouse.

Snowpark Python (shown, not run in this notebook):

```python
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, sum as sum_

session = Session.builder.configs(connection_parameters).create()

orders = session.table("RAW.ORDERS.ORDERS")
daily = orders.group_by(col("ORDER_DATE")).agg(sum_(col("AMOUNT")).alias("REVENUE"))
daily.write.mode("overwrite").save_as_table("ANALYTICS.MARTS.DAILY_REVENUE")
```

The SQL that Snowflake runs for it is roughly:

```sql
CREATE OR REPLACE TABLE analytics.marts.daily_revenue AS
SELECT order_date, SUM(amount) AS revenue
FROM raw.orders.orders
GROUP BY order_date;
```

## UDFs and stored procedures

### Explanation
SQL alone cannot express every rule or every multi-step job, so Snowflake lets you add your own code in two forms.

A user-defined function (UDF) returns one value per row, like a built-in function. A scalar UDF must be side-effect free and cannot query tables. That restriction lets the optimizer inline it or run it in parallel across a scan.

A stored procedure is a named program you start with `CALL`. It runs its statements in order and can contain DML, DDL and control flow such as loops and `IF`. By default it runs with the privileges of its owner role. Its statements are not one automatic transaction: only what you wrap in an explicit transaction commits or rolls back together.

```text
UDF:        value in  --> value out      (per row, inside a query)
Procedure:  CALL      --> many statements (orchestration)
```

Python UDFs and procedures are packaged with a runtime version, a handler that names the function to call, optional `PACKAGES`, and optional `IMPORTS` for staged files. Snowflake loads the staged code into the execution environment.

A simple rule: a UDF computes, a procedure coordinates. Neither should loop over rows one by one.

### Use cases
- **Shared business logic:** a `fiscal_quarter(date)` SQL UDF used in every report.
- **Idempotent daily load:** a procedure deletes one day and re-inserts it, called by a task.
- **Python helpers:** a Python UDF with `PACKAGES` for logic that SQL cannot express.

### Limitations and things to remember
- Scalar UDFs cannot query tables and must have no side effects.
- A row-by-row Python UDF is far slower than a native SQL expression.
- A procedure's statements are not one transaction unless you open one explicitly.
- Procedures run with owner's rights by default.
- Staged code must be uploaded before `CREATE FUNCTION` and re-uploaded when it changes.
- Looping over rows in a procedure multiplies statement overhead; keep the loop inside one set-based statement or batch the work.

### Code example

The code creates a SQL UDF used in a query and a Snowflake Scripting procedure that reloads one day.

Snowflake SQL (shown, not run in this notebook):

```sql
-- SQL scalar UDF: pure, composable, safe inside a large scan.
CREATE OR REPLACE FUNCTION analytics.fiscal_quarter(d DATE)
  RETURNS STRING LANGUAGE SQL
AS $$
  CASE WHEN MONTH(d) >= 2 THEN 'Q' || CEIL((MONTH(d) - 1) / 3.0)::INT ELSE 'Q4' END
$$;

SELECT analytics.fiscal_quarter(order_date) AS fq, SUM(amount)
FROM analytics.orders GROUP BY fq;

-- Procedure: several statements, idempotent for one load date.
CREATE OR REPLACE PROCEDURE analytics.load_daily(load_date DATE)
  RETURNS STRING LANGUAGE SQL
AS $$
BEGIN
  DELETE FROM analytics.marts.daily_revenue WHERE order_date = :load_date;
  INSERT INTO analytics.marts.daily_revenue
    SELECT order_date, SUM(amount) FROM raw.orders.orders
    WHERE order_date = :load_date GROUP BY order_date;
  RETURN 'loaded ' || :load_date::STRING;
END;
$$;

CALL analytics.load_daily('2026-10-04');
```

## Roles and privileges (RBAC)

### Explanation
Granting rights to each person on each table does not scale and is impossible to audit. Snowflake uses role-based access control (RBAC) instead. A privilege is a right on an object, such as `SELECT` on a table. A role is a named bundle of privileges. You grant privileges to roles, and roles to users or to other roles. A user acts through the role active in their session.

The predefined roles split responsibility:

```text
ACCOUNTADMIN   account settings, sees everything
 ├── SYSADMIN       owns warehouses and databases
 └── SECURITYADMIN  manages grants, can monitor any object
       └── USERADMIN   creates users and roles
PUBLIC         granted to every user automatically
```

Because `SYSADMIN` and `SECURITYADMIN` are granted to `ACCOUNTADMIN`, it inherits broad access rather than being granted it object by object.

Ownership is separate from privileges. Exactly one role owns an object, and `OWNERSHIP` implies every privilege on it.

A future grant is a grant stored on a schema that applies to objects created later in it. It is not retroactive. The usual pattern is a functional role (such as `analytics_engineer`) with the privileges, granted up to `SYSADMIN` so admins keep control, and granted to people.

### Use cases
- **Team access:** one `analytics_engineer` role reads all marts; new hires just get the role.
- **New tables in deployments:** future grants make tables created by CI readable without extra grants.
- **Access audits:** query `GRANTS_TO_ROLES` to list what each role can do.
- **Service accounts:** a task-owner role with only the privileges the pipeline needs.

### Limitations and things to remember
- Future grants do not reach objects that already exist; issue them before the deployment, or grant explicitly afterward.
- Exactly one role owns an object; `OWNERSHIP` implies every privilege.
- Running applications or tasks as `ACCOUNTADMIN` makes least-privilege auditing impossible.
- Transfer ownership to custom roles and grant those roles to `SYSADMIN`.
- Keep `ACCOUNTADMIN` for account administration only.
- Grant roles to people, not privileges.

### Code example

The code creates a functional role, grants it read access including future tables, and audits it.

Snowflake SQL (shown, not run in this notebook):

```sql
USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS analytics_engineer;

USE ROLE SYSADMIN;
GRANT USAGE ON WAREHOUSE transform_wh TO ROLE analytics_engineer;
GRANT USAGE ON DATABASE analytics TO ROLE analytics_engineer;
GRANT USAGE ON SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT SELECT ON ALL TABLES IN SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT SELECT ON FUTURE TABLES IN SCHEMA analytics.marts TO ROLE analytics_engineer;
GRANT ROLE analytics_engineer TO ROLE SYSADMIN;   -- keep the hierarchy intact

-- Give a person the role, not the privileges.
GRANT ROLE analytics_engineer TO USER analyst_jane;

-- Audit.
SHOW GRANTS TO ROLE analytics_engineer;
SELECT * FROM snowflake.account_usage.grants_to_roles
WHERE role LIKE 'ANALYTICS%' AND deleted_on IS NULL;
```

## Secure views, row access policies and masking

### Explanation
If every query must remember to filter rows or hide PII, one forgotten `WHERE` clause leaks data. Snowflake moves that enforcement into the engine.

A secure view is a view whose definition is hidden from users who do not own it. A row access policy is a function attached to a table that returns true or false per row, deciding which rows a role can see. A masking policy is a function attached to a column that decides what value a role sees, for example `***@example.com` instead of the real email.

Both policies are evaluated at query time against the role active in the session, not stored per user or per tool. The same SQL run by two roles can return different rows or values. That is why enforcement survives a client that forgets a filter.

```text
table scan --> row access policy (which rows exist?) --> masking policy (what do values look like?) --> result
```

A row access policy usually compares `CURRENT_ROLE()` with a small mapping table. A masking policy can use `CURRENT_ROLE()` and `CURRENT_USER()`. `CURRENT_ROLE()` is the safer key, because roles are the unit of governance and users change. Because a policy is a function called as part of the scan, it must be deterministic enough for the engine to plan.

### Use cases
- **Regional sales data:** each regional role sees only its own region's orders.
- **PII protection:** only `PII_ADMIN` sees real customer emails; others see a masked value.
- **Hiding test rows:** a secure view filters out test orders and hides its logic.

### Limitations and things to remember
- Row access and masking policies require Enterprise Edition.
- A policy that reads a mapping table adds work to every query on the protected object; keep the mapping small and the logic simple.
- A policy that returns null can behave differently from false; write explicit boolean expressions.
- Rows are filtered first, then values are masked on the surviving rows.
- Key policies on `CURRENT_ROLE()` rather than users.

### Code example

The code creates a secure view, a region-based row access policy and an email masking policy.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE SECURE VIEW analytics.v_orders AS
SELECT order_id, customer_id, amount
FROM analytics.orders WHERE order_status <> 'test';

-- Rows: a role sees only regions mapped to it.
CREATE OR REPLACE ROW ACCESS POLICY security.region_policy
  AS (region STRING) RETURNS BOOLEAN ->
    CURRENT_ROLE() = 'ACCOUNTADMIN'
    OR EXISTS (SELECT 1 FROM security.region_map m
               WHERE m.role_name = CURRENT_ROLE() AND m.region = region);
ALTER TABLE analytics.orders ADD ROW ACCESS POLICY security.region_policy ON (region);

-- Columns: mask the email user name for everyone but PII admins.
CREATE OR REPLACE MASKING POLICY security.email_mask
  AS (val STRING) RETURNS STRING ->
    CASE WHEN CURRENT_ROLE() IN ('PII_ADMIN', 'ACCOUNTADMIN') THEN val
         ELSE REGEXP_REPLACE(val, '^[^@]+', '***') END;
ALTER TABLE analytics.customers MODIFY COLUMN email SET MASKING POLICY security.email_mask;
```

## Data sharing

### Explanation
Sending data to a partner usually means exports, file drops and sync jobs that are always out of date. Snowflake data sharing removes the copy. A share is a named set of grants on objects in a database. The provider (the account that owns the data) grants privileges to the share and adds consumer accounts.

The consumer creates a database from the share. That database is a read-only mount that resolves to the provider's storage and metadata. No rows are copied when the share is created, and no synchronization job exists. New rows the provider commits are visible to the consumer on their next query, limited by the provider's grants and any Time Travel the provider applies.

```text
provider account                       consumer account
analytics.marts.fct_orders  <--share-- shared_sales (read-only mount, consumer's warehouse)
```

The consumer needs `IMPORTED PRIVILEGES` on the mounted database and pays for the compute that reads it. A share is region-scoped. To share across regions or clouds, the provider first replicates the database, and replication is serverless and billed separately. For consumers without a Snowflake account, the provider can create and manage a reader account.

Treat a share like a product with a schema contract: consumers depend on its shape.

### Use cases
- **Partner reporting:** give a retailer live access to its own order facts without file exports.
- **Internal multi-account setups:** share a central mart to business-unit accounts.
- **Cross-region customers:** replicate the database, then share from the secondary region.
- **Non-Snowflake consumers:** serve a small partner through a provider-managed reader account.

### Limitations and things to remember
- Shared data is read-only for the consumer; no copy, no ETL.
- Consumers pay for the compute they use to query the share.
- Cross-region or cross-cloud sharing needs replication first; replication is billed separately.
- A share can expose object definitions as well as data.
- Replacing or dropping the shared database breaks every consumer; coordinate changes and prefer additive schema evolution.

### Code example

The code shows the provider creating and granting a share, the consumer mounting it, and the replication step for cross-region use.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Provider: grant objects to the share, then add consumers.
CREATE SHARE sales_share;
GRANT USAGE ON DATABASE analytics TO SHARE sales_share;
GRANT USAGE ON SCHEMA analytics.marts TO SHARE sales_share;
GRANT SELECT ON TABLE analytics.marts.fct_orders TO SHARE sales_share;
ALTER SHARE sales_share ADD ACCOUNTS = consumer_org.consumer_acct;

-- Consumer: mount the share and grant access to a local role.
CREATE DATABASE shared_sales FROM SHARE provider_org.provider_acct.sales_share;
GRANT IMPORTED PRIVILEGES ON DATABASE shared_sales TO ROLE analyst;
SELECT COUNT(*) FROM shared_sales.marts.fct_orders;

-- Cross-region: replicate first, then share from the secondary.
ALTER DATABASE analytics ENABLE REPLICATION TO ACCOUNTS secondary_org.secondary_acct;
```

## Transactions and isolation

### Explanation
A transaction groups statements so they all commit or all roll back, which keeps data consistent when several jobs write at once. Snowflake transactions are ACID: atomic, consistent, isolated and durable.

Isolation decides what one transaction sees of another's changes. Snowflake's default is read committed at the statement level: each statement takes a fresh snapshot (a consistent view of committed data) when it starts. So two statements in the same explicit transaction can see different versions if a commit lands between them. Readers do not block writers and writers do not block readers, because a reader just reads the versions in its snapshot. Concurrent DML on the same table is serialized by the transaction manager, not through a visible lock wait.

Outside an explicit transaction, each statement is its own transaction and commits on success; this is autocommit. `BEGIN` opens a transaction that stays open until `COMMIT` or `ROLLBACK`.

```text
BEGIN
  UPDATE ...     -- part of the transaction
  CREATE TABLE   -- DDL: commits on its own, immediately
ROLLBACK         -- undoes the UPDATE, not the CREATE
```

`SERIALIZABLE` isolation changes conflict handling. Instead of letting statements interleave, Snowflake aborts one of the conflicting transactions, so the application must retry.

### Use cases
- **Money-style transfers:** debit one row and credit another atomically.
- **Delete-and-reload:** replace one partition of a mart inside one transaction so readers never see it empty.
- **Strict concurrent updates:** use `SERIALIZABLE` with retry logic where interleaving is unacceptable.

### Limitations and things to remember
- Default isolation is statement-level read committed, not repeatable read.
- DDL inside a transaction commits immediately and independently; `ROLLBACK` does not undo it.
- `SERIALIZABLE` aborts one conflicting transaction; build retries.
- A long open transaction keeps old versions alive, delays storage cleanup and can consume the Time Travel window.
- Keep transactions short: wrap only the statements that must be atomic.

### Code example

The code runs an atomic transfer under SERIALIZABLE and shows that DDL survives a rollback.

Snowflake SQL (shown, not run in this notebook):

```sql
ALTER SESSION SET TRANSACTION_ISOLATION_LEVEL = 'SERIALIZABLE';

BEGIN;
UPDATE analytics.accounts SET balance = balance - 100 WHERE account_id = 1;
UPDATE analytics.accounts SET balance = balance + 100 WHERE account_id = 2;
COMMIT;

-- DDL auto-commits: the ROLLBACK does not undo the CREATE.
BEGIN;
CREATE TABLE analytics.tmp_check (id NUMBER);
ROLLBACK;
SHOW TABLES LIKE 'TMP_CHECK' IN SCHEMA analytics;   -- the table still exists
```

## Resource monitors

### Explanation
Snowflake bills compute in credits, and a runaway query or a warehouse left running can burn a month's budget. A resource monitor turns cost from something you notice later into an enforced limit. It is an object with a credit quota for an interval (for example monthly) and triggers: actions at percentages of the quota. `NOTIFY` sends an alert, `SUSPEND` stops the warehouse after running queries finish, and `SUSPEND_IMMEDIATE` also cancels running queries.

A monitor totals the credits used by its assigned warehouses plus the cloud services that supported them. The interval resets at 12:00 AM UTC. You can assign a monitor to individual warehouses or to the whole account.

```text
0% ──── 75% NOTIFY ──── 100% SUSPEND ──── 110% SUSPEND_IMMEDIATE
```

Some compute is outside a monitor's control. An account-level monitor does not control Snowflake-managed (serverless) compute for Snowpipe, automatic clustering or materialized views; budgets exist for those. A warehouse-level monitor can observe cloud-services credits but cannot suspend them. A suspension clears when the interval resets, the quota is raised, or the warehouse is unassigned.

Set quotas from measured usage in `WAREHOUSE_METERING_HISTORY`, not guesses.

### Use cases
- **ELT budget cap:** a monthly 2,000-credit monitor on the ELT warehouse with early notification.
- **Account guardrail:** an account monitor catches warehouses that have no monitor of their own.
- **Sizing quotas:** daily credit totals per warehouse set realistic thresholds.

### Limitations and things to remember
- Only `ACCOUNTADMIN` can create a resource monitor.
- A warehouse can belong to at most one warehouse monitor; an account monitor does not override it, and either can trigger a suspend.
- Serverless features (Snowpipe, automatic clustering, materialized views) are not controlled; use budgets.
- Thresholds count all cloud-services credits, ignoring the 10 percent daily adjustment, so a monitor can show 100 percent while the bill is smaller.
- Suspension takes effect slightly after the threshold; for a hard limit, suspend below quota, for example at 90 percent.

### Code example

The code creates a warehouse monitor and an account monitor, then checks real credit usage.

Snowflake SQL (shown, not run in this notebook):

```sql
CREATE OR REPLACE RESOURCE MONITOR elt_monitor WITH CREDIT_QUOTA = 2000
  FREQUENCY = MONTHLY
  TRIGGERS ON 75 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND
           ON 110 PERCENT DO SUSPEND_IMMEDIATE;
ALTER WAREHOUSE elt_wh SET RESOURCE_MONITOR = elt_monitor;

-- Account-level guardrail.
CREATE OR REPLACE RESOURCE MONITOR account_monitor WITH CREDIT_QUOTA = 20000
  FREQUENCY = MONTHLY
  TRIGGERS ON 90 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
ALTER ACCOUNT SET RESOURCE_MONITOR = account_monitor;

-- Credits used per warehouse per day, last 14 days.
SELECT warehouse_name, DATE_TRUNC('day', start_time) AS usage_day, SUM(credits_used) AS credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -14, CURRENT_TIMESTAMP())
GROUP BY 1, 2 ORDER BY usage_day DESC, credits DESC;
```

## Migrating to Snowflake

### Explanation
Moving from Redshift, Hadoop or another warehouse is mostly about semantics and validation, not syntax. The safe order is data first, then logic: load and reconcile the data, then move the transformations that depend on it.

Translate in two passes. The first pass is mechanical syntax, which tooling such as SnowConvert handles. The second pass is semantics, which no tool decides for you. Empty string versus null, integer division and rounding, timestamp precision and time zones, identifier case folding, and sequence or `IDENTITY` behavior all differ across engines. Each difference needs a test with real data, not just a code review.

Physical design does not carry over either:

```text
Redshift DISTKEY / SORTKEY  -> load ordering + optional clustering key
Hive partition columns      -> directory layout; PARTITION BY for external/Iceberg tables
Indexes, ANALYZE            -> nothing: micro-partition statistics are automatic
Cluster shape               -> separate warehouse sizing decision
```

A micro-partition is the block of rows Snowflake stores and manages for you, with statistics it keeps automatically.

Reconciliation proves the migration worked. Compare more than row counts: sums, distinct keys, null rates and report-level results. Keep the source system running until a full reporting cycle passes on both sides.

### Use cases
- **Redshift lift-and-shift:** translate SQL with tooling, then redesign sort and distribution keys as clustering.
- **Hadoop/Hive retirement:** move partitioned tables to Iceberg or native tables.
- **Cutover sign-off:** a reconciliation log per subject area shows what is done and what to investigate.

### Limitations and things to remember
- Tools translate syntax; you own the data model, permissions and workload placement.
- Unquoted identifiers fold to uppercase in Snowflake; code that assumed lowercase can break even when the SQL is valid.
- Null versus empty string, integer division, timestamp precision and `IDENTITY` behavior need data tests.
- Secondary indexes and `ANALYZE` have no equivalent.
- Row counts alone are not reconciliation; also check sums, distinct keys and reports.
- Keep the source running until a full reporting cycle passes.

### Code example

The code reconciles a migrated table on several measures and reports status per subject area.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Reconcile beyond row counts: sums, distinct keys and null rates.
SELECT 'orders' AS entity,
       COUNT(*)                                             AS rows_loaded,
       COUNT(DISTINCT order_id)                             AS distinct_keys,
       SUM(amount)                                          AS total_amount,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customers
FROM analytics.fct_orders;

-- Progress per subject area.
SELECT subject_area, status, source_rows, target_rows,
       CASE WHEN source_rows = target_rows THEN 'ok' ELSE 'investigate' END AS check_result
FROM migration.reconciliation_log
ORDER BY subject_area;
```

## Gen2 standard warehouses

### Explanation
Analytics and data-engineering jobs keep getting bigger, and teams want them to finish sooner on the same kind of warehouse. Gen2 is Snowflake's answer. A Gen2 standard warehouse is a newer generation of the standard virtual warehouse. It runs on faster hardware and adds software optimizations aimed at analytics and data engineering.

You rarely have to ask for it. If you create a standard warehouse and do not name a generation, Snowflake now creates Gen2 wherever the region supports it. An older warehouse converts in place with `ALTER WAREHOUSE ... SET GENERATION = '2'`.

Faster hardware is not automatically cheaper. A Gen2 warehouse can bill at a different credit rate, so the real question is whether the job finishes enough sooner to pay for that rate. Think of it like a faster delivery van: you only save money if it does more drops per hour than it costs extra per hour.

```text
CREATE WAREHOUSE (no GENERATION) --> Gen2 if the region supports it
existing Gen1 warehouse --ALTER ... SET GENERATION = '2'--> Gen2
```

### Use cases
- **New transformation warehouse:** create it without a generation and get Gen2 by default in supported regions.
- **Slow nightly ELT:** convert the transform warehouse to Gen2 and compare run time and credits against last week.
- **Fleet modernisation:** benchmark one representative workload per warehouse before converting the whole fleet.

### Limitations and things to remember
- Gen2 is the default for new standard warehouses only where the region supports it.
- Gen2 is not offered in the X5LARGE and X6LARGE sizes.
- Check the current credit rates before you convert; a faster warehouse can still cost more per job.
- Benchmark a representative workload first, not a single hand-picked query.
- Conversion uses `ALTER WAREHOUSE ... SET GENERATION = '2'`; nothing about your SQL changes.

### Code example

The code creates a Gen2 standard warehouse and converts an existing one.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Gen2 standard warehouse (the default for new standard warehouses where available)
CREATE OR REPLACE WAREHOUSE transform_wh
  WAREHOUSE_SIZE = 'MEDIUM'
  GENERATION = '2'
  AUTO_SUSPEND = 60;

-- Convert an existing warehouse after benchmarking it
ALTER WAREHOUSE legacy_wh SET GENERATION = '2';
```

## Cortex AI functions

### Explanation
Data engineers often need to label, summarise or translate free text, such as support tickets or reviews. Before, that meant exporting the text to a separate machine-learning service and loading the answers back. Cortex AI functions remove that round trip. They are SQL functions that call hosted large language models (LLMs, models that read and write text) from inside a query.

The family includes `AI_COMPLETE`, `AI_CLASSIFY`, `AI_FILTER`, `AI_AGG`, `AI_EXTRACT`, `AI_SENTIMENT`, `AI_SUMMARIZE_AGG`, `AI_EMBED`, `AI_TRANSLATE`, `AI_PARSE_DOCUMENT` and others. You call them like any other function, for example `AI_SENTIMENT(body)`.

The catch is cost. You pay according to the amount of text processed, not the number of rows returned. So the shape of a good Cortex pipeline is: filter the input, remove duplicates, run the function once, and save the result in a table. Reports then read that table. Treat the model like an expensive translator you pay by the page: send each page once and keep the translation.

```text
raw text --filter + dedupe--> small set --AI_*()--> results table <-- dashboards read here
```

### Use cases
- **Ticket triage:** score the sentiment of yesterday's support tickets and store it in a table.
- **Routing:** use `AI_CLASSIFY` to tag inbound emails by team before loading them.
- **Document intake:** pull fields out of PDFs with `AI_PARSE_DOCUMENT` and `AI_EXTRACT`.
- **Search features:** build vector embeddings with `AI_EMBED` for similarity lookups.

### Limitations and things to remember
- Cost grows with the amount of text processed.
- Run the functions over a filtered, deduplicated set, not a whole raw table.
- Persist the results; do not call AI functions from a dashboard query.
- Common mistake: calling an AI function row by row inside a frequently refreshed view, which pays for the same text on every refresh.

### Code example

The code scores one day of tickets once and saves the answers so later queries do not pay again.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Cortex AI function over a filtered set, results persisted once
CREATE OR REPLACE TABLE analytics.ticket_sentiment AS
SELECT ticket_id, AI_SENTIMENT(body) AS sentiment
FROM raw.support_tickets
WHERE created_at >= DATEADD(day, -1, CURRENT_TIMESTAMP());

-- Dashboards read the stored result, not the model
SELECT sentiment, COUNT(*) FROM analytics.ticket_sentiment GROUP BY 1;
```

## Horizon Catalog, Open Catalog and Apache Polaris

### Explanation
Apache Iceberg is an open table format: table data and metadata live in your own cloud storage, so several engines can share the same tables. To do that safely, every engine needs one catalog. A catalog is the service that records which metadata file is the current version of each table. The Iceberg REST catalog is a standard HTTP API for that service.

Three names come up, and they are easy to mix up:

```text
Apache Polaris    open-source Iceberg REST catalog, open-sourced by Snowflake
       |          (Apache top-level project since February 2026)
       v
Open Catalog      Snowflake's managed Polaris service
Horizon Catalog   Snowflake's catalog, now the one recommended for new customers;
                  exposes an Iceberg REST API over Snowflake-managed Iceberg tables
```

The problem Horizon solves is catalog sync. Before, sharing a Snowflake-managed Iceberg table with Spark meant keeping a second catalog in step. Because Horizon speaks the Iceberg REST API itself, external engines such as Spark, Flink and Dremio can read and write Snowflake-managed Iceberg tables directly. One catalog means one source of truth for the current table version.

### Use cases
- **Shared lakehouse:** let Spark jobs write to the same Iceberg tables that Snowflake queries, through Horizon.
- **Streaming into Iceberg:** point a Flink job at Horizon's Iceberg REST endpoint instead of a separately synced catalog.
- **Multi-engine BI:** let Dremio read Snowflake-managed Iceberg tables without copying data.
- **Platform choice:** pick Horizon for a new account; recognise Open Catalog or self-run Polaris in existing estates.

### Limitations and things to remember
- Apache Polaris is the open-source project; Open Catalog is the managed Polaris service.
- Polaris became an Apache top-level project in February 2026.
- Snowflake now steers new customers to Horizon Catalog rather than Open Catalog.
- Horizon's benefit is no separate catalog sync for Snowflake-managed Iceberg tables.
- Iceberg tables themselves are covered in their own section of this handbook.

### Code example

The code creates a Snowflake-managed Iceberg table, the kind Horizon exposes to external engines through its Iceberg REST API.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Snowflake-managed Iceberg table: data in your bucket, catalog = Snowflake (Horizon)
CREATE OR REPLACE ICEBERG TABLE lake.orders (
  order_id NUMBER,
  amount   NUMBER(12,2)
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'lake_vol'
  BASE_LOCATION = 'orders/';
-- Spark, Flink or Dremio then connect to Horizon's Iceberg REST endpoint
-- and read or write lake.orders without a second, synced catalog.
```

## Hybrid tables

### Explanation
Standard Snowflake tables are column-oriented. They are great at scanning millions of rows, but slow and awkward for "read or update this one row now". Pipelines still need that kind of small, fast state, for example "when did this pipeline last succeed?". Hybrid tables, part of Snowflake's Unistore feature, fill that gap.

A hybrid table is a row-oriented table type built for low-latency point reads and writes. A point read fetches one row by its key. Hybrid tables live next to your analytical tables, so you can join them in the same query.

They behave more like an operational database than a warehouse table:
- The primary key is required and enforced.
- Unique and foreign keys are enforced too.
- Secondary indexes (extra lookup structures on non-key columns) are maintained synchronously, so they are always up to date.

The trade-off is scan speed. A hybrid table is like an index card box: perfect for finding one card, slow for counting every word on every card. Keep aggregations in standard tables.

### Use cases
- **Pipeline checkpoints:** store the last processed timestamp per pipeline and update it after each run.
- **Run status:** look up a job's current status by ID from an orchestrator.
- **Application metadata:** keep small config or entity tables that need enforced keys.

### Limitations and things to remember
- Generally available in AWS and Azure commercial regions; in preview on Google Cloud.
- A primary key is mandatory and enforced; unique and foreign keys are enforced as well.
- They take more storage than standard tables.
- They fit small operational state, not bulk analytics.
- Common mistake: using a hybrid table to speed up analytics; aggregations over millions of rows still belong in standard tables.

### Code example

The code creates a hybrid table for pipeline state with an enforced key and a secondary index, then does a point update.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Hybrid table: enforced primary key plus a secondary index
CREATE OR REPLACE HYBRID TABLE ops.pipeline_state (
  pipeline_id VARCHAR PRIMARY KEY,
  last_run_ts TIMESTAMP_NTZ,
  status      VARCHAR,
  INDEX idx_status (status)
);

-- Point write by key: the access pattern hybrid tables are built for
UPDATE ops.pipeline_state
SET last_run_ts = CURRENT_TIMESTAMP(), status = 'SUCCEEDED'
WHERE pipeline_id = 'orders_daily';
```

## Reading the query profile

### Explanation
When a query is slow, the risk is fixing the wrong thing. The query profile prevents that. It is the operator tree Snowflake shows for one executed query: each node is an operator (a step such as TableScan, Join, Aggregate or Sort) with its share of execution time, rows and bytes.

Start with the numbers in `QUERY_HISTORY`, then confirm in the profile. Compare them one pair at a time:

```text
QUEUED_OVERLOAD_TIME vs EXECUTION_TIME   -> concurrency problem or SQL problem?
PARTITIONS_SCANNED / PARTITIONS_TOTAL    -> is pruning working?
PERCENTAGE_SCANNED_FROM_CACHE            -> warm or cold cache?
BYTES_SPILLED_TO_LOCAL/REMOTE_STORAGE    -> did it run out of memory?
```

Pruning means skipping micro-partitions (the compressed blocks Snowflake stores data in) that cannot match the filter. Only when queuing is negligible and the scan is already small is the remaining time a pure operator-cost problem.

`TOTAL_ELAPSED_TIME` includes compilation and queuing. A query can look slow on a dashboard while its SQL runs fast, so subtract those pieces before you rewrite anything.

Read the tree bottom-up. A parent's elapsed time includes its children's, so one slow child makes every ancestor look slow. Find the operator with the largest share of execution time, fix that node's input, then read the profile again. The operator with the most rows is not always the one to tune.

### Use cases
- **Dashboard complaint:** check whether a "slow" query was really queued or compiling, not executing.
- **Pruning check:** confirm a date filter actually skips partitions before adding a clustering key.
- **Before a rewrite:** find the one operator that dominates instead of guessing.

### Limitations and things to remember
- Operator-level detail exists only in the profile of a single `query_id`.
- `TOTAL_ELAPSED_TIME` = queuing + compilation + execution; compare `EXECUTION_TIME` for SQL cost.
- A slow child inflates every ancestor's time; read bottom-up.
- Most rows does not mean most time.
- `SYSTEM$EXPLAIN_PLAN_JSON` returns the plan, not just the timings.

### Code example

The code finds the slowest recent SELECTs with their diagnostic columns, then pulls one query's text and plan.

Snowflake SQL (shown, not run in this notebook):

```sql
-- 1. Worst offenders in the last day, with the columns to compare
SELECT query_id, warehouse_name, warehouse_size,
       total_elapsed_time / 1000   AS elapsed_s,
       queued_overload_time / 1000 AS queued_s,
       compilation_time / 1000     AS compile_s,
       execution_time / 1000       AS exec_s,
       ROUND(partitions_scanned / NULLIF(partitions_total, 0) * 100, 1) AS pct_scanned,
       percentage_scanned_from_cache,
       bytes_spilled_to_local_storage,
       bytes_spilled_to_remote_storage
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND query_type = 'SELECT'
ORDER BY total_elapsed_time DESC
LIMIT 25;

-- 2. One query in detail, including its text
SELECT query_id, query_text, execution_time, bytes_scanned
FROM snowflake.account_usage.query_history
WHERE query_id = '<query_id>';

-- 3. The plan, not just the timings
SELECT SYSTEM$EXPLAIN_PLAN_JSON('<query_id>');
```

## Tuning a slow query, step by step

### Explanation
Tuning by instinct wastes time and credits. A repeatable loop works better:

```text
1. find      QUERY_HISTORY: slowest queries by EXECUTION_TIME
2. classify  queueing / remote spill / local spill / poor pruning / operator cost
3. confirm   query profile of that one query_id
4. change    exactly one thing
5. compare   re-run and read the same columns again
```

Classification uses a simple order of checks. If the query waited in the queue longer than it ran, it is a concurrency problem. Otherwise, remote spill beats local spill, which beats poor pruning (more than half the partitions scanned). If none of those apply, the cost is in the operators themselves.

There are two sources of history, with different freshness. `SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY` is a view that holds a year of history but can lag by up to about 45 minutes. The `QUERY_HISTORY` table function in `INFORMATION_SCHEMA` is much closer to real time. Use the account view for trends and attribution, and the table function for the statement you just ran while iterating.

Filter before you aggregate. A sweep over every query in a day reads many rows and buries the signal. Start from the workload's warehouse and a narrow time window, exclude internal statements, and only then sort by `EXECUTION_TIME` or by spill.

### Use cases
- **Morning triage:** classify every query over 60 seconds from yesterday by likely cause.
- **Iterating on a rewrite:** use the `INFORMATION_SCHEMA` table function to see the run you just did.
- **Warehouse review:** filter to one warehouse and rank its queries by spill.

### Limitations and things to remember
- `ACCOUNT_USAGE.QUERY_HISTORY` keeps a year of history but lags by up to about 45 minutes.
- Change one thing at a time, or you cannot tell what helped.
- Operator-level attribution exists only in the profile of the single `query_id`.
- `QUEUED_PROVISIONING_TIME` (waiting for the warehouse to start) is separate from `QUEUED_OVERLOAD_TIME`.
- The triage rule is a starting point; the profile is the proof.

### Code example

The code classifies yesterday's long-running queries by their most likely cause.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Triage: classify the dominant symptom per query (queries over 60 s)
SELECT query_id, warehouse_name,
       total_elapsed_time / 1000 AS elapsed_s,
       CASE
         WHEN queued_overload_time > execution_time           THEN 'concurrency: queueing'
         WHEN bytes_spilled_to_remote_storage > 0             THEN 'memory: remote spill'
         WHEN bytes_spilled_to_local_storage > 0              THEN 'memory: local spill'
         WHEN partitions_scanned > partitions_total * 0.5     THEN 'io: poor pruning'
         ELSE 'cpu: operator cost'
       END AS likely_cause
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND execution_time > 60000
ORDER BY execution_time DESC
LIMIT 50;

-- While iterating, read the near-real-time table function instead
SELECT query_id, execution_time, bytes_spilled_to_local_storage
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT => 10));
```

The runnable cell below applies the same triage `CASE` to a small simulated `query_history` table in DuckDB.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE query_history AS
    SELECT * FROM (VALUES
        ('q1', 90000, 30000,             0,             0,  100, 1000),
        ('q2',  1000, 80000, 5000000000,  2000000000,  100, 1000),
        ('q3',     0, 70000,  800000000,             0,  100, 1000),
        ('q4',     0, 65000,             0,             0,  900, 1000),
        ('q5',     0, 61000,             0,             0,   20, 1000)
    ) AS t(query_id, queued_overload_time, execution_time,
           bytes_spilled_to_local_storage, bytes_spilled_to_remote_storage,
           partitions_scanned, partitions_total)
""")

triage = dict(con.execute("""
    SELECT query_id,
           CASE
             WHEN queued_overload_time > execution_time       THEN 'concurrency: queueing'
             WHEN bytes_spilled_to_remote_storage > 0         THEN 'memory: remote spill'
             WHEN bytes_spilled_to_local_storage > 0          THEN 'memory: local spill'
             WHEN partitions_scanned > partitions_total * 0.5 THEN 'io: poor pruning'
             ELSE 'cpu: operator cost'
           END AS likely_cause
    FROM query_history
    WHERE execution_time > 60000 OR queued_overload_time > 60000
""").fetchall())
print(triage)

assert triage["q1"] == "concurrency: queueing"
assert triage["q2"] == "memory: remote spill"
assert triage["q3"] == "memory: local spill"
assert triage["q4"] == "io: poor pruning"
assert triage["q5"] == "cpu: operator cost"

## Spilling to local and remote storage

### Explanation
Every warehouse has a fixed amount of memory per node. Spilling happens when an operation needs more memory than that and has to write intermediate data out. Local spill writes to the node's SSD and slows the query down. Remote spill goes over the network to object storage, is an order of magnitude slower, and is a severe slowdown.

```text
memory  --full-->  local SSD (slow)  --full-->  remote object storage (severe)
```

Any operator that builds a hash table or sorts its full input can spill: `JOIN`, `GROUP BY`, `ORDER BY`, `DISTINCT` and window functions. Spilling is reported per operator and summed per query. A query can have several spilling operators, so the query total can hide which one dominates; the profile shows it per node.

The fix order matters. First reduce what must be held in memory: filter and aggregate before the join, drop unneeded sorts, remove a fan-out join on a non-unique key. That shrinks the structure that must fit and also cuts the work of every operator downstream. Only then raise the warehouse size, which gives more memory per query.

A bigger warehouse can hide a structural problem until the data grows. If a query spills again after one size increase, stop resizing and reshape the query.

### Use cases
- **Slow nightly MERGE:** find it in the top spillers list and pre-aggregate the source before the join.
- **Systematic spill on a warehouse:** sum spill by warehouse over 7 days to decide whether to resize it.
- **Exploding join:** deduplicate a non-unique join key before joining.

### Limitations and things to remember
- Local spill is a slowdown; remote spill is roughly an order of magnitude worse.
- Spill columns: `BYTES_SPILLED_TO_LOCAL_STORAGE` and `BYTES_SPILLED_TO_REMOTE_STORAGE`.
- Reduce rows before the spilling operator first; resize second.
- If it spills again after one resize, reshape the query instead of resizing again.
- The query-level total can hide which operator spills; check the profile.

### Code example

The code finds the biggest spillers, then shows the before and after of a query that spilled because of a fan-out join.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Biggest spillers in the last day
SELECT query_id, warehouse_name, warehouse_size,
       ROUND(bytes_spilled_to_local_storage  / POWER(1024, 3), 2) AS gb_local,
       ROUND(bytes_spilled_to_remote_storage / POWER(1024, 3), 2) AS gb_remote
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
  AND bytes_spilled_to_local_storage + bytes_spilled_to_remote_storage > 0
ORDER BY bytes_spilled_to_remote_storage DESC, bytes_spilled_to_local_storage DESC
LIMIT 25;

-- Before: join on a non-unique key, late filter, unnecessary sort
SELECT o.customer_id, SUM(o.amount) AS revenue
FROM analytics.orders o
JOIN analytics.events e ON e.customer_id = o.customer_id
WHERE o.order_date >= '2026-01-01'
GROUP BY o.customer_id
ORDER BY revenue DESC;

-- After: filter and aggregate before the join, deduplicate the key, drop the sort
WITH recent_orders AS (
  SELECT customer_id, SUM(amount) AS revenue
  FROM analytics.orders
  WHERE order_date >= '2026-01-01'
  GROUP BY customer_id
),
distinct_customers AS (
  SELECT DISTINCT customer_id FROM analytics.events
)
SELECT r.customer_id, r.revenue
FROM recent_orders r
JOIN distinct_customers c ON c.customer_id = r.customer_id;

-- Only if spill persists: more memory per query
ALTER WAREHOUSE transform_wh SET WAREHOUSE_SIZE = 'LARGE';
```

The runnable cell below measures, in DuckDB, how many rows the join produces before and after the rewrite. Fewer rows into the join is what keeps the hash table in memory. It also shows that the "before" query inflates revenue, because each order is counted once per matching event.

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE orders AS
    SELECT i AS order_id, i % 1000 AS customer_id, 10.0 AS amount,
           DATE '2025-07-01' + CAST(i % 365 AS INTEGER) AS order_date
    FROM range(20000) t(i)
""")
con.execute("""
    CREATE TABLE events AS
    SELECT i AS event_id, i % 1000 AS customer_id FROM range(50000) t(i)
""")

before_join_rows = con.execute("""
    SELECT COUNT(*) FROM orders o JOIN events e ON e.customer_id = o.customer_id
    WHERE o.order_date >= DATE '2026-01-01'
""").fetchone()[0]
after_join_rows = con.execute("""
    WITH recent_orders AS (
        SELECT customer_id, SUM(amount) AS revenue FROM orders
        WHERE order_date >= DATE '2026-01-01' GROUP BY customer_id),
    distinct_customers AS (SELECT DISTINCT customer_id FROM events)
    SELECT COUNT(*) FROM recent_orders r JOIN distinct_customers c USING (customer_id)
""").fetchone()[0]
print(f"rows out of the join: before={before_join_rows:,} after={after_join_rows:,}")

before_revenue = con.execute("""
    SELECT SUM(o.amount) FROM orders o JOIN events e ON e.customer_id = o.customer_id
    WHERE o.order_date >= DATE '2026-01-01'
""").fetchone()[0]
true_revenue = con.execute(
    "SELECT SUM(amount) FROM orders WHERE order_date >= DATE '2026-01-01'"
).fetchone()[0]

assert after_join_rows <= 1000
assert before_join_rows > 100 * after_join_rows
assert before_revenue == 50 * true_revenue  # 50 events per customer inflate every order

## Query patterns that make Snowflake slow

### Explanation
Snowflake applies many optimizations for free. The optimizer pushes filters down to the scan, so it can prune micro-partitions, and it chooses join order from table statistics. Most slow queries are slow because the SQL hides information the optimizer needs.

The common culprits:
- **A function on the filter column**, such as `YEAR(order_date) = 2026`, hides the literal range, so pruning is lost. Write a raw range instead.
- **`SELECT *`** sends unused columns through every operator.
- **`UNION`** deduplicates and sorts; `UNION ALL` does not.
- **`ORDER BY` without `LIMIT`** sorts the whole result when you only need the top rows.
- **An implicit type conversion**, such as joining a number to a VARCHAR, prevents a clean join plan. Cast once, before the join.
- **`DISTINCT` where the grain is already unique** adds a needless aggregation.
- **A `LIKE` with a leading wildcard** and **repeated VARIANT extraction** also hide or repeat work.

Rewriting for the optimizer means making the predicate more literal and the plan more obvious, not making the query shorter.

Repeated work is a pattern too: recomputing a stable aggregate on every dashboard refresh, re-scanning a table that changed in one partition, or extracting the same JSON path in every query. Before rewriting, ask whether the answer could be cached, materialized, or read from a clustered or optimized access path.

### Use cases
- **Code review of dbt models:** flag functions on filter columns and `SELECT *` before merge.
- **Year-to-date reports:** replace `YEAR(col) = ...` with a half-open date range.
- **Stacking yearly tables:** use `UNION ALL` when the inputs cannot overlap.
- **Semi-structured events:** promote hot JSON fields into a typed view once.

### Limitations and things to remember
- `YEAR(order_date) = 2026` and `order_date >= '2026-01-01' AND order_date < '2027-01-01'` return the same rows, but only the second prunes.
- `UNION` is not free: it deduplicates and sorts.
- Implicit VARCHAR-to-number joins block a clean join plan; cast explicitly once.
- A `LIKE '%x'` cannot use pruning.
- Cache or materialize stable results instead of tuning a query that should not re-run.

### Code example

The code shows each slow pattern next to its rewrite.

Snowflake SQL (shown, not run in this notebook):

```sql
-- 1. Function on the filter column loses pruning; use a raw range
SELECT COUNT(*) FROM analytics.orders WHERE YEAR(order_date) = 2026;
SELECT COUNT(*) FROM analytics.orders
WHERE order_date >= '2026-01-01' AND order_date < '2027-01-01';

-- 2. Name the columns instead of SELECT *
SELECT order_id, customer_id, amount FROM analytics.orders WHERE order_date = '2026-10-04';

-- 3. UNION ALL when the inputs cannot overlap
SELECT customer_id FROM analytics.orders_2025
UNION ALL
SELECT customer_id FROM analytics.orders_2026;

-- 4. Bound the sort when you only need the top rows
SELECT customer_id, SUM(amount) AS revenue FROM analytics.orders
GROUP BY customer_id ORDER BY revenue DESC LIMIT 100;

-- 5. Cast explicitly once, before the join (not an implicit VARCHAR-to-number join)
SELECT * FROM analytics.orders o
JOIN (SELECT TO_NUMBER(customer_id_str) AS customer_id FROM analytics.customers) c
  ON o.customer_id = c.customer_id;

-- 6. No DISTINCT when order_id is already unique
SELECT order_id, customer_id FROM analytics.orders;

-- 7. Promote a repeated VARIANT path into a typed view once
CREATE OR REPLACE VIEW analytics.v_events AS
  SELECT payload:user.id::NUMBER AS user_id FROM raw.events;
SELECT user_id, COUNT(*) FROM analytics.v_events GROUP BY 1;
```

The runnable cell below checks two rewrites with DuckDB: the function-free range returns the same count as `YEAR()`, and `UNION` versus `UNION ALL` shows the deduplication work `UNION` does. Timings are printed for interest; only the results are asserted, because timings vary by machine.

In [ ]:
import time

import duckdb

con = duckdb.connect()
con.execute("""
    CREATE TABLE orders AS
    SELECT i AS order_id, i % 5000 AS customer_id,
           DATE '2024-01-01' + CAST(i % 1096 AS INTEGER) AS order_date
    FROM range(2000000) t(i)
    ORDER BY order_date
""")


def timed(sql: str) -> tuple[int, float]:
    start = time.perf_counter()
    value = con.execute(sql).fetchone()[0]
    return value, time.perf_counter() - start


year_count, year_s = timed("SELECT COUNT(*) FROM orders WHERE YEAR(order_date) = 2025")
range_count, range_s = timed(
    "SELECT COUNT(*) FROM orders "
    "WHERE order_date >= DATE '2025-01-01' AND order_date < DATE '2026-01-01'"
)
print(f"YEAR(): {year_count:,} rows in {year_s:.4f}s | range: {range_count:,} rows in {range_s:.4f}s")
assert year_count == range_count

union_rows, _ = timed(
    "SELECT COUNT(*) FROM (SELECT customer_id FROM orders WHERE order_date < DATE '2025-01-01' "
    "UNION SELECT customer_id FROM orders WHERE order_date >= DATE '2025-01-01')"
)
union_all_rows, _ = timed(
    "SELECT COUNT(*) FROM (SELECT customer_id FROM orders WHERE order_date < DATE '2025-01-01' "
    "UNION ALL SELECT customer_id FROM orders WHERE order_date >= DATE '2025-01-01')"
)
print(f"UNION keeps {union_rows:,} rows; UNION ALL keeps {union_all_rows:,} rows")
assert union_rows == 5000
assert union_all_rows == 2000000

## Loading data fast

### Explanation
Slow loads usually come from file shape, not from the warehouse. Load throughput is bounded by the slowest of three things:

```text
1. how many files can be processed in parallel   <- hurt by very small files
2. how fast each file can be read                <- hurt by tiny files and by one huge file
3. how fast the warehouse writes partitions
```

So aim for compressed files of 100 to 250 MB. Compacting small files before a load and setting `MAX_FILE_SIZE` when unloading attack the same bottleneck.

`COPY INTO` is the bulk load command. A few options make it fast and safe. `PATTERN` loads a bounded slice of files in one batch. `VALIDATION_MODE = 'RETURN_ERRORS'` is a dry run that reports bad rows without loading anything. `ON_ERROR` decides deliberately what happens to bad rows. `PURGE = TRUE` removes files after a successful load.

The two ingestion paths bill differently. `COPY INTO` runs on a warehouse you manage, so you control its size and lifecycle, and large scheduled batches let it suspend between runs. Snowpipe and Snowpipe Streaming run on Snowflake-managed serverless compute, billed as credits attributed to the pipe or the streaming client. That suits continuous arrival but is less predictable per byte. Choose Snowpipe for latency and batch `COPY` for cost.

### Use cases
- **Daily Parquet drop:** load one day's files with `PATTERN` and `MATCH_BY_COLUMN_NAME`.
- **New vendor feed:** dry-run with `VALIDATION_MODE` before the first real load.
- **Exports for partners:** unload into many ~100 MB files instead of one giant file.
- **Load monitoring:** track files, rows and errors per table per day in `COPY_HISTORY`.

### Limitations and things to remember
- Target 100 to 250 MB compressed files; very small or single huge files slow the load.
- `VALIDATION_MODE` loads nothing; it only returns errors.
- `ON_ERROR = 'CONTINUE'` skips bad rows silently unless you check `COPY_HISTORY`.
- `FORCE = FALSE` (the default) skips files already loaded.
- Snowpipe is convenient for continuous data but less predictable per byte than a batch warehouse.

### Code example

The code dry-runs and then loads one day of Parquet files, and checks the load results in `COPY_HISTORY`.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Dry run before committing to a new format or feed
COPY INTO raw.orders.orders FROM @raw.orders.landing
  FILE_FORMAT = (TYPE = PARQUET)
  VALIDATION_MODE = 'RETURN_ERRORS';

-- Load a bounded slice, tolerate bad rows, clean up after success
COPY INTO raw.orders.orders
FROM @raw.orders.landing
FILE_FORMAT = (TYPE = PARQUET)
MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
PATTERN = '.*2026-10-04.*[.]parquet'
ON_ERROR = 'CONTINUE'
PURGE = TRUE
FORCE = FALSE;

-- Unload into many appropriately sized files instead of one giant file
COPY INTO @raw.orders.exports/daily/
FROM (SELECT * FROM analytics.marts.daily_revenue)
FILE_FORMAT = (TYPE = PARQUET)
MAX_FILE_SIZE = 100000000;

-- Files that failed in the last week, and why
SELECT file_name, status, first_error_message, last_load_time
FROM snowflake.account_usage.copy_history
WHERE status = 'LOAD_FAILED'
  AND last_load_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY last_load_time DESC;
```

## Tuning without burning credits

### Explanation
Tuning experiments cost credits too: every test run, resize and clustering key is paid for. And a change that makes a query faster can still make it more expensive. So judge every change by credits per query, not by latency alone.

Attribution makes a tuning session provable. A query tag is a label you set on the session with `ALTER SESSION SET QUERY_TAG`. Every query in that session carries it, so `QUERY_ATTRIBUTION_HISTORY` can group the before runs and the after runs. `WAREHOUSE_METERING_HISTORY` shows the warehouse-level effect per hour. Without a tag, the comparison is a guess against everything else that ran in the same hour.

Do the free checks first:

```text
Tuning checklist
1. Is the query queued rather than slow? Check QUEUED_OVERLOAD_TIME.
2. Is it pruning? Compare PARTITIONS_SCANNED with PARTITIONS_TOTAL.
3. Is it spilled? Local spill is slow, remote spill is severe.
4. Are functions or casts on the filter column?
5. Is SELECT * pulling columns nothing uses?
6. Is the warehouse actually suspending between runs?
7. Did credits per query improve, not just latency?
8. Can the result be cached or precomputed instead of re-run?
```

Only after those come changes that cost something to test, such as a clustering key or a resize. Run those against a resource monitor with a low daily quota. A resource monitor is a credit budget that can notify you or suspend a warehouse when usage crosses a threshold.

### Use cases
- **Proving a rewrite paid off:** tag before and after runs and compare credits per query.
- **Sandboxed experiments:** attach a 50-credit daily monitor to a dedicated tuning warehouse.
- **Idle cost hunt:** check hourly metering to see whether a warehouse really suspends.

### Limitations and things to remember
- Judge by credits per query, not latency alone.
- Without a `QUERY_TAG`, before and after cannot be separated from other workload.
- Resource monitor triggers can `NOTIFY` (e.g. at 80 percent) and `SUSPEND` (e.g. at 100 percent).
- Free checks (queuing, pruning, spill, suspend) come before paid ones (clustering, resize).
- Keep `AUTO_SUSPEND` short (e.g. 60 seconds) on a tuning warehouse.

### Code example

The code tags a tuning session, compares credits per tag, and caps the experiment with a resource monitor.

Snowflake SQL (shown, not run in this notebook):

```sql
-- Tag the session so before/after runs can be grouped
ALTER SESSION SET QUERY_TAG = 'tune:pruning_2026_10';

-- Credits per query and per tag: the number that decides whether tuning paid
SELECT query_tag,
       COUNT(*) AS queries,
       SUM(credits_used) AS credits,
       ROUND(SUM(credits_used) / NULLIF(COUNT(*), 0), 6) AS credits_per_query
FROM snowflake.account_usage.query_attribution_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY query_tag
ORDER BY credits DESC;

-- Warehouse-level effect per hour
SELECT warehouse_name, DATE_TRUNC('hour', start_time) AS hour,
       SUM(credits_used_compute) AS compute_credits,
       SUM(credits_used_cloud_services) AS cloud_credits
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -1, CURRENT_TIMESTAMP())
GROUP BY 1, 2
ORDER BY hour DESC;

-- Guardrails while experimenting
CREATE OR REPLACE RESOURCE MONITOR tune_monitor WITH CREDIT_QUOTA = 50
  FREQUENCY = DAILY
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
ALTER WAREHOUSE tune_wh SET RESOURCE_MONITOR = tune_monitor
  AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
```

## Tuning exercises

These use only the standard library and model tuning arithmetic in the units the platform reports. Treat each assert as a check on the formula.

### Try it 1. Is this a pruning problem?

**Task.** Given the metrics for one query, compute the scanned-partition ratio and decide whether the query has a pruning problem. A query is considered adequately pruned when it scans at most half of the partitions.

**Hint.** `ratio = partitions_scanned / partitions_total`; a ratio above 0.5 means pruning is poor.

**Expected result.** `0.960 (96.0%)` scanned and the verdict `fix pruning`.

In [ ]:
metrics = {"partitions_scanned": 4800, "partitions_total": 5000}

ratio = metrics["partitions_scanned"] / metrics["partitions_total"]
verdict = "fix pruning" if ratio > 0.5 else "pruning is acceptable"

print(f"scanned ratio = {ratio:.3f} ({ratio * 100:.1f}%)")
print("verdict:", verdict)

assert round(ratio, 3) == 0.96
assert verdict == "fix pruning"

### Try it 2. Scale up or scale out?

**Task.** Using a credits-per-hour table, compute the credits consumed by a 600-second query on a MEDIUM warehouse and by the same query taking 300 seconds on a LARGE warehouse. Then compute the cost of running three SMALL clusters for a full hour and compare it with one MEDIUM cluster for a full hour.

**Hint.** `credits = credits_per_hour * seconds / 3600`. Doubling the size should roughly halve the runtime at about the same credits.

**Expected result.** The MEDIUM/600s and LARGE/300s runs each cost about `0.667` credits, so scale-up is credit-neutral. Three SMALL clusters cost `6.0` credits per hour versus `4.0` for one MEDIUM.

In [ ]:
CREDITS_PER_HOUR = {"XSMALL": 1, "SMALL": 2, "MEDIUM": 4, "LARGE": 8, "XLARGE": 16}


def credits(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600


medium_run = credits("MEDIUM", 600)
large_run = credits("LARGE", 300)
print(f"MEDIUM 600s = {medium_run:.3f} credits")
print(f"LARGE  300s = {large_run:.3f} credits")
assert abs(medium_run - large_run) < 1e-9

scale_out = 3 * credits("SMALL", 3600)
scale_up = 1 * credits("MEDIUM", 3600)
print(f"3 x SMALL for 1h = {scale_out:.1f} credits")
print(f"1 x MEDIUM for 1h = {scale_up:.1f} credits")
assert scale_out == 6.0 and scale_up == 4.0

### Try it 3. Detect remote spill

**Task.** Write a function that classifies a metrics dictionary as `no spill`, `local spill`, or `remote spill`, then classify three example queries. Remote spill is the severe case because it means local disk was exhausted too.

**Hint.** Check `bytes_spilled_to_remote_storage` first; it takes precedence over local spill. Use `.get(key, 0)` so missing columns behave as zero.

**Expected result.** The three examples classify as `local spill`, `remote spill`, and `no spill`.

In [ ]:
def classify_spill(metrics):
    local = metrics.get("bytes_spilled_to_local_storage", 0)
    remote = metrics.get("bytes_spilled_to_remote_storage", 0)
    if remote > 0:
        return "remote spill"
    if local > 0:
        return "local spill"
    return "no spill"


local_only = {"bytes_spilled_to_local_storage": 2_000_000_000,
              "bytes_spilled_to_remote_storage": 0}
remote_too = {"bytes_spilled_to_local_storage": 1_000_000_000,
              "bytes_spilled_to_remote_storage": 500_000_000}
clean = {"bytes_spilled_to_local_storage": 0, "bytes_spilled_to_remote_storage": 0}

for name, m in [("query_a", local_only), ("query_b", remote_too), ("query_c", clean)]:
    print(name, "->", classify_spill(m))

assert classify_spill(local_only) == "local spill"
assert classify_spill(remote_too) == "remote spill"
assert classify_spill(clean) == "no spill"

### Try it 4. Choose a clustering key

**Task.** Given the approximate cardinality of four columns, pick a two-column clustering key that leads with the date used by range filters and then uses the highest-cardinality filtered column.

**Hint.** Keep the range column first, then take the column with the largest distinct count among the remaining candidates. `max(iterable, key=...)` selects it in one pass.

**Expected result.** The chosen key is `('order_date', 'customer_id')`.

In [ ]:
cardinality = {
    "order_date": 90,          # low cardinality, but range filters use it
    "customer_id": 2_000_000,  # high cardinality, also filtered
    "country": 40,             # nobody filters on it
    "event_type": 12,
}
range_column = "order_date"

id_column = max((c for c in cardinality if c != range_column), key=cardinality.get)
cluster_key = (range_column, id_column)
print("cluster key:", cluster_key)
print(f"{id_column} has {cardinality[id_column]:,} distinct values")

assert cluster_key == ("order_date", "customer_id")

### Try it 5. Multi-cluster scaling: STANDARD versus ECONOMY

**Task.** Simulate twelve ticks of arriving queries on a multi-cluster warehouse with at most three clusters. STANDARD adds a cluster as soon as queries queue; ECONOMY adds one only after three consecutive ticks with a non-empty queue. Compare total queue length with total cluster-ticks.

**Hint.** Each tick: finish work, add arrivals, apply the scaling policy, then start as many queries as there is spare capacity. Sum the queue length and the cluster count at the end of each tick.

**Expected result.** STANDARD keeps the queue at zero but pays 35 cluster-ticks; ECONOMY pays 29 cluster-ticks and leaves a queue of 14 query-ticks.

In [ ]:
def simulate(policy, ticks=12, arrivals_per_tick=1, runtime=3, max_clusters=3):
    clusters = 1
    active = []          # remaining ticks for each running query
    queue = 0
    queue_ticks = 0
    cluster_ticks = 0
    patient = 0          # consecutive ticks with a non-empty queue (ECONOMY only)
    for _ in range(ticks):
        active = [r - 1 for r in active if r - 1 > 0]
        queue += arrivals_per_tick
        if policy == "STANDARD":
            if queue > 0:
                clusters = min(max_clusters, clusters + 1)
        else:
            if queue > 0:
                patient += 1
                if patient % 3 == 0:
                    clusters = min(max_clusters, clusters + 1)
            else:
                patient = 0
        capacity = max(clusters - len(active), 0)
        started = min(capacity, queue)
        active += [runtime] * started
        queue -= started
        queue_ticks += queue
        cluster_ticks += clusters
    return queue_ticks, cluster_ticks


standard = simulate("STANDARD")
economy = simulate("ECONOMY")
print(f"STANDARD: queue_ticks={standard[0]:2d}  cluster_ticks={standard[1]}")
print(f"ECONOMY : queue_ticks={economy[0]:2d}  cluster_ticks={economy[1]}")

assert standard[0] < economy[0], "STANDARD should queue less"
assert standard[1] > economy[1], "STANDARD should pay for more clusters"

### Try it 6. Estimate cost per query

**Task.** Given a price of 3.00 currency units per credit, compute the credits and the cost of four queries and the average cost per query.

**Hint.** Cost is `credits * price`, and credits are `credits_per_hour * seconds / 3600`.

**Expected result.** Total cost is about `1.72`, an average of about `0.43` per query.

In [ ]:
PRICE_PER_CREDIT = 3.00
CREDITS_PER_HOUR = {"SMALL": 2, "MEDIUM": 4, "LARGE": 8, "XLARGE": 16}
queries = [("SMALL", 12), ("MEDIUM", 30), ("LARGE", 120), ("XLARGE", 60)]


def query_cost(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600 * PRICE_PER_CREDIT


costs = []
for size, seconds in queries:
    cost = query_cost(size, seconds)
    costs.append(cost)
    print(f"{size:<7} {seconds:>4}s -> {cost:.4f}")

total = sum(costs)
print(f"total = {total:.4f}, average = {total / len(costs):.4f}")
assert round(total, 2) == 1.72
assert round(total / len(costs), 2) == 0.43

### Try it 7. Read a cache-hit column

**Task.** Given the `percentage_scanned_from_cache` values for five runs of the same query, report the average and decide whether most runs were cache-served. Treat a run as cache-served when the value is above 90 percent.

**Hint.** Count the runs above the threshold and compare with half the number of runs.

**Expected result.** The average is `63.0` percent and most runs (3 of 5) were cache-served.

In [ ]:
scanned_from_cache = [98.5, 12.0, 99.1, 8.4, 97.0]

average = sum(scanned_from_cache) / len(scanned_from_cache)
cache_served = [value for value in scanned_from_cache if value > 90]
mostly_cached = len(cache_served) >= len(scanned_from_cache) / 2

print(f"average from cache = {average:.1f}%")
print(f"cache-served runs = {len(cache_served)} of {len(scanned_from_cache)}")
print("mostly cached:", mostly_cached)

assert round(average, 1) == 63.0
assert len(cache_served) == 3
assert mostly_cached is True

### Try it 8. Decide whether a scale-up experiment paid off

**Task.** A query took 60 seconds on a SMALL warehouse and 32 seconds on a MEDIUM warehouse. Compute the credits for both, the speedup, and decide whether the change is worth keeping. Treat it as worth keeping when the credits rise by no more than 10 percent and the speedup is at least 1.5x.

**Hint.** Compare credits, not size names, and remember that the smaller warehouse is not cheaper per credit-hour.

**Expected result.** Speedup is about `1.88x`, credits rise about `6.7` percent, so the scale-up is worth keeping.

In [ ]:
CREDITS_PER_HOUR = {"SMALL": 2, "MEDIUM": 4, "LARGE": 8}


def credits(size, seconds):
    return CREDITS_PER_HOUR[size] * seconds / 3600


before = credits("SMALL", 60)
after = credits("MEDIUM", 32)
speedup = 60 / 32
credit_change = (after - before) / before
worth_keeping = credit_change <= 0.10 and speedup >= 1.5

print(f"SMALL 60s  = {before:.4f} credits")
print(f"MEDIUM 32s = {after:.4f} credits")
print(f"speedup = {speedup:.2f}x, credit change = {credit_change * 100:+.1f}%")
print("worth keeping:", worth_keeping)

assert round(speedup, 2) == 1.88
assert round(credit_change * 100, 1) == 6.7
assert worth_keeping is True

## Real world use cases for Snowflake

### 1. Retail ELT with dbt and incremental loads

A retailer lands point-of-sale, e-commerce, and inventory feeds in S3, loads them with `COPY INTO`, and transforms them with dbt models. Daily volume is roughly 400 million order lines and 2 TB of raw files, which makes file sizing, incremental models, and pruning the difference between a pipeline that finishes before the business day and one that does not.

```sql
COPY INTO raw.orders.orders FROM @raw.orders.landing/order_date=2026-10-04/
  FILE_FORMAT = (TYPE = PARQUET) MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
  PATTERN = '.*[.]parquet' ON_ERROR = 'CONTINUE' PURGE = TRUE;

MERGE INTO analytics.fct_orders AS t
USING stg_orders AS s ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET amount = s.amount, status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT (order_id, customer_id, order_date, amount, status, updated_at)
VALUES (s.order_id, s.customer_id, s.order_date, s.amount, s.status, s.updated_at);

ALTER TABLE analytics.fct_orders CLUSTER BY (order_date, customer_id);
```

Why: loading and transformation use separate warehouses, the merge makes retries safe, and clustering on `(order_date, customer_id)` keeps month-over-month reporting scans small.

### 2. Fintech regulatory reporting with Time Travel

A licensed lender must reproduce balances and classifications exactly as they stood on a reporting date. Loads are append-only and immutable, and every reporting query pins a timestamp so the number in the report can be regenerated months later by anyone with the same privileges.

```sql
CREATE TABLE analytics.regulatory_snapshot CLONE analytics.fct_balances
  AT (TIMESTAMP => '2026-09-30 23:59:59'::TIMESTAMP);

SELECT customer_id,
       SUM(CASE WHEN balance >= 10000 THEN balance ELSE 0 END) AS reportable_balance
FROM analytics.regulatory_snapshot
GROUP BY customer_id;

ALTER TABLE analytics.fct_balances SET DATA_RETENTION_TIME_IN_DAYS = 90;
```

Why: Time Travel plus clones make point-in-time reporting a query rather than a restore exercise, and secure views with masking policies keep PII out of the analyst role.

### 3. Multi-tenant SaaS analytics with row access policies

A SaaS vendor serves every customer from one copy of the data but must guarantee tenant isolation. A mapping table links each role to the tenant ids it may see, and a row access policy on the fact tables enforces the filter for every consumer, including BI tools that never add a tenant predicate of their own.

```sql
CREATE OR REPLACE ROW ACCESS POLICY security.tenant_policy
  AS (tenant_id STRING) RETURNS BOOLEAN ->
    CURRENT_ROLE() IN ('ACCOUNTADMIN', 'TENANT_ADMIN')
    OR EXISTS (
      SELECT 1 FROM security.tenant_map m
      WHERE m.role_name = CURRENT_ROLE() AND m.tenant_id = tenant_id
    );

ALTER TABLE analytics.fct_usage ADD ROW ACCESS POLICY security.tenant_policy ON (tenant_id);
```

Why: isolation is enforced by the engine rather than by each dashboard, so a misconfigured report cannot leak another tenant's data and one copy serves everyone.

### 4. Streaming CDC with Snowpipe Streaming and streams

A payments platform emits change events to Kafka, a connector writes them through Snowpipe Streaming into a raw table, a stream captures the delta, and a task merges it into the serving table every few minutes. End-to-end latency stays under ten minutes without any file landing step, which matters because payment status is customer-visible.

```sql
CREATE OR REPLACE TASK analytics.apply_payments
  WAREHOUSE = transform_wh
  SCHEDULE = '5 MINUTE'
  WHEN SYSTEM$STREAM_HAS_DATA('raw.payments.payments_stream')
AS
  MERGE INTO analytics.fct_payments AS t
  USING (
    SELECT payment_id, status, amount, updated_at, METADATA$ACTION AS action
    FROM raw.payments.payments_stream
    QUALIFY ROW_NUMBER() OVER (PARTITION BY payment_id ORDER BY updated_at DESC) = 1
  ) AS s
  ON t.payment_id = s.payment_id
  WHEN MATCHED AND s.action = 'DELETE' THEN DELETE
  WHEN MATCHED THEN UPDATE SET status = s.status, amount = s.amount
  WHEN NOT MATCHED AND s.action = 'INSERT'
    THEN INSERT (payment_id, status, amount) VALUES (s.payment_id, s.status, s.amount);

ALTER TASK analytics.apply_payments RESUME;
```

Why: Snowpipe Streaming removes file-landing latency, the stream makes each task incremental, and the merge makes a retried task safe.

### 5. Cost-optimization program with resource monitors

A platform team runs dozens of warehouses and must cut spend without hurting SLAs. The program starts with attribution: credits per warehouse, per user, and per query tag, so the conversation moves from opinions to numbers. Then it right-sizes the biggest consumers, shortens auto-suspend on batch warehouses, and removes full refreshes that a `MERGE` or a dynamic table would do incrementally.

```sql
ALTER SESSION SET QUERY_TAG = 'team:marketing_pipeline';

SELECT warehouse_name,
       SUM(credits_used) AS credits,
       ROUND(SUM(credits_used) * 3.00, 2) AS approx_usd
FROM snowflake.account_usage.warehouse_metering_history
WHERE start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY warehouse_name
ORDER BY credits DESC;

CREATE OR REPLACE RESOURCE MONITOR marketing_monitor WITH CREDIT_QUOTA = 500
  FREQUENCY = MONTHLY
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND;
```

Why: credits are attributable per warehouse, user, and query tag, so the program is measured rather than argued about; typical wins are shorter auto-suspend, right-sizing, fewer full refreshes, and separated workload compute.

### 6. Lakehouse with Iceberg tables shared across engines

A media company keeps one copy of its event data as Iceberg on S3. Snowflake writes the curated tables, Spark reads and backfills them, and Trino serves ad-hoc queries against the same Parquet files and metadata. No engine is locked out and the organization stops paying to copy the same data into a warehouse-native format for every consumer.

```sql
CREATE ICEBERG TABLE analytics.events (
  event_id STRING, event_date DATE, customer_id NUMBER, payload VARIANT)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'iceberg_vol'
  BASE_LOCATION = 'analytics/events';

INSERT INTO analytics.events
SELECT event_id, event_date, customer_id, payload
FROM raw.events_staging
WHERE event_date = '2026-10-04';
```

Why: the table is open, so no engine is locked out and the organization stops paying to copy the same data into a warehouse-native format for every consumer, at the cost of making write ownership, maintenance, and catalog choice explicit.